In [ ]:
%pip install pandas openpyxl scikit-learn matplotlib torch transformers datasets accelerate

In [ ]:
import pandas as pd

file_path = "data/synthetic_b2b_transactions_2000.xlsx"

df = pd.read_excel(file_path)

print("Form:", df.shape)
display(df.head())

In [ ]:
# Die eine Excel-Spalte anhand des Semikolons in einzelne Spalten aufteilen
df_split = df.iloc[:, 0].str.split(";", expand=True)

# Die ursprüngliche Kopfzeile ebenfalls aufteilen
column_names = df.columns[0].split(";")
df_split.columns = column_names

print("Neue Form:", df_split.shape)
print("Anzahl Spalten:", len(df_split.columns))

display(df_split[["description", "parentCategoryName", "categoryName"]].head(10))

In [ ]:
# Relevante Daten für die Klassifikation auswählen
data = df_split[
    ["description", "parentCategoryName", "categoryName"]
].copy()

# Leere Werte entfernen
data = data.dropna()

print("Anzahl verwendbarer Transaktionen:", len(data))
print("Anzahl Oberkategorien:", data["parentCategoryName"].nunique())
print("Anzahl Unterkategorien:", data["categoryName"].nunique())

print("\nVerteilung der Unterkategorien:")
display(data["categoryName"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Unterkategorien in numerische Labels umwandeln
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["categoryName"])

# 80 % Training, 10 % Validation, 10 % Test
train_data, temp_data = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    stratify=data["label"]
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    stratify=temp_data["label"]
)

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Anzahl Klassen:", len(label_encoder.classes_))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "ProsusAI/finbert"
num_labels = len(label_encoder.classes_)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

print("FinBERT erfolgreich geladen.")
print("Anzahl Zielklassen:", num_labels)

In [ ]:
# Texte für FinBERT vorbereiten
train_encodings = tokenizer(
    train_data["description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_data["description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_data["description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("Tokenisierung abgeschlossen.")
print("Training:", len(train_encodings["input_ids"]))
print("Validation:", len(val_encodings["input_ids"]))
print("Test:", len(test_encodings["input_ids"]))

In [ ]:
import torch
from torch.utils.data import Dataset

class TransactionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = TransactionDataset(
    train_encodings,
    train_data["label"].tolist()
)

val_dataset = TransactionDataset(
    val_encodings,
    val_data["label"].tolist()
)

test_dataset = TransactionDataset(
    test_encodings,
    test_data["label"].tolist()
)

print("PyTorch-Datensätze erstellt.")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Verwendetes Gerät:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Keine CUDA-GPU erkannt – Training läuft über die CPU.")

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

# Klassengewichte berechnen:
# seltene Kategorien bekommen ein höheres Gewicht
classes = np.arange(len(label_encoder.classes_))

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_data["label"].to_numpy()
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

print("Klassengewichte berechnet.")
print("Anzahl Gewichte:", len(class_weights))


# Weighted Focal Loss 
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        return focal_loss.mean()


criterion = FocalLoss(
    alpha=class_weights,
    gamma=2.0
)

print("Weighted Focal Loss erfolgreich erstellt.")

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

# Einstellungen
batch_size = 16
epochs = 3
learning_rate = 2e-5

# Daten portionsweise bereitstellen
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

# Modell auf CPU/GPU verschieben
model.to(device)

# Optimizer
optimizer = AdamW(
    model.parameters(),
    lr=learning_rate
)

# Training
for epoch in range(epochs):

    print(f"\n===== Epoche {epoch + 1}/{epochs} =====")

    model.train()
    total_train_loss = 0

    progress_bar = tqdm(train_loader, desc="Training")

    for batch in progress_bar:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(outputs.logits, labels)
            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = accuracy_score(all_labels, all_predictions)

print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Prüfen, ob identische Transaktionsbeschreibungen
# gleichzeitig in Training und Test vorkommen

train_descriptions = set(
    train_data["description"].str.strip().str.lower()
)

test_descriptions = set(
    test_data["description"].str.strip().str.lower()
)

duplicates = train_descriptions.intersection(test_descriptions)

print("Identische Beschreibungen in Train und Test:", len(duplicates))

if len(duplicates) > 0:
    print("\nBeispiele:")
    for text in list(duplicates)[:20]:
        print("-", text)
else:
    print("Keine identischen Beschreibungen gefunden.")

In [ ]:
import re

def clean_description(text):
    text = str(text).lower().strip()

    # Datumsangaben wie 2025-07 entfernen
    text = re.sub(r"\b20\d{2}[-/.]\d{1,2}\b", " ", text)

    # Rechnungs-/Referenznummern entfernen
    text = re.sub(r"\b[a-z]{1,4}-?\d{3,}\b", " ", text)

    # Sonderzeichen entfernen
    text = re.sub(r"[^a-zäöüß\s]", " ", text)

    # Mehrfache Leerzeichen entfernen
    text = re.sub(r"\s+", " ", text).strip()

    return text


data["clean_description"] = data["description"].apply(clean_description)

display(
    data[["description", "clean_description", "categoryName"]].head(20)
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF-Repräsentation der bereinigten Beschreibungen
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(
    data["clean_description"]
)

# Cosine Similarity zwischen allen Beschreibungen
similarity_matrix = cosine_similarity(tfidf_matrix)

print("TF-IDF-Matrix:", tfidf_matrix.shape)
print("Similarity-Matrix:", similarity_matrix.shape)

In [ ]:
# Ähnliche Beschreibungen anzeigen
similar_pairs = []

threshold = 0.85

for i in range(len(data)):
    for j in range(i + 1, len(data)):
        similarity = similarity_matrix[i, j]

        if similarity >= threshold:
            similar_pairs.append((
                data.iloc[i]["clean_description"],
                data.iloc[j]["clean_description"],
                data.iloc[i]["categoryName"],
                data.iloc[j]["categoryName"],
                similarity
            ))

print("Gefundene ähnliche Paare:", len(similar_pairs))

# Erste 30 Beispiele anzeigen
for pair in similar_pairs[:30]:
    print("\nText 1:", pair[0])
    print("Text 2:", pair[1])
    print("Kategorie 1:", pair[2])
    print("Kategorie 2:", pair[3])
    print("Ähnlichkeit:", round(pair[4], 3))

In [ ]:
# Exakt gleiche bereinigte Beschreibungen bekommen dieselbe Gruppe
data["group_id"] = data.groupby("clean_description").ngroup()

print("Anzahl Transaktionen:", len(data))
print("Anzahl eindeutiger Gruppen:", data["group_id"].nunique())

# Größte Gruppen anzeigen
display(
    data.groupby(
        ["group_id", "clean_description", "categoryName"]
    )
    .size()
    .reset_index(name="anzahl")
    .sort_values("anzahl", ascending=False)
    .head(20)
)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 80 % Train / 20 % Temp anhand der group_id
gss1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, temp_idx = next(
    gss1.split(
        data,
        groups=data["group_id"]
    )
)

train_grouped = data.iloc[train_idx].copy()
temp_grouped = data.iloc[temp_idx].copy()

# 20 % Temp nochmal halbieren -> 10 % Validation / 10 % Test
gss2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    gss2.split(
        temp_grouped,
        groups=temp_grouped["group_id"]
    )
)

val_grouped = temp_grouped.iloc[val_idx].copy()
test_grouped = temp_grouped.iloc[test_idx].copy()

print("Training:", len(train_grouped))
print("Validation:", len(val_grouped))
print("Test:", len(test_grouped))

print("\nGruppen:")
print("Training:", train_grouped["group_id"].nunique())
print("Validation:", val_grouped["group_id"].nunique())
print("Test:", test_grouped["group_id"].nunique())

# Kontrolle: keine Gruppe darf mehrfach vorkommen
train_groups = set(train_grouped["group_id"])
val_groups = set(val_grouped["group_id"])
test_groups = set(test_grouped["group_id"])

print("\nÜberschneidungen:")
print("Train/Val:", len(train_groups & val_groups))
print("Train/Test:", len(train_groups & test_groups))
print("Val/Test:", len(val_groups & test_groups))

In [ ]:
all_categories = set(data["categoryName"])

train_categories = set(train_grouped["categoryName"])
val_categories = set(val_grouped["categoryName"])
test_categories = set(test_grouped["categoryName"])

print("Kategorien gesamt:", len(all_categories))
print("Training:", len(train_categories))
print("Validation:", len(val_categories))
print("Test:", len(test_categories))

print("\nFehlen im Training:")
print(all_categories - train_categories)

print("\nFehlen in Validation:")
print(all_categories - val_categories)

print("\nFehlen im Test:")
print(all_categories - test_categories)

In [ ]:
groups_per_category = (
    data[["categoryName", "group_id"]]
    .drop_duplicates()
    .groupby("categoryName")
    .size()
    .sort_values()
)

print("Anzahl unterschiedlicher Gruppen pro Kategorie:\n")
print(groups_per_category)

print("\nKategorien mit weniger als 3 Gruppen:")
print(groups_per_category[groups_per_category < 3])

In [ ]:
for category, count in groups_per_category.items():
    print(f"{category}: {count}")

In [ ]:
import numpy as np

# Eine Tabelle mit genau einer Zeile pro Grundmuster erstellen
group_table = (
    data.groupby(["group_id", "categoryName"], as_index=False)
    .first()
)

rng = np.random.RandomState(42)

train_groups = []
val_groups = []
test_groups = []

for category, cat_groups in group_table.groupby("categoryName"):
    
    ids = cat_groups["group_id"].tolist()
    rng.shuffle(ids)

    n = len(ids)

    if n == 1:
        # Nur ein Muster vorhanden -> muss ins Training
        train_groups.extend(ids)

    elif n == 2:
        # Ein Muster Training, eines Validation
        train_groups.append(ids[0])
        val_groups.append(ids[1])

    else:
        # Mindestens 3 Muster:
        # eines Validation, eines Test, Rest Training
        val_groups.append(ids[0])
        test_groups.append(ids[1])
        train_groups.extend(ids[2:])


train_grouped = data[data["group_id"].isin(train_groups)].copy()
val_grouped = data[data["group_id"].isin(val_groups)].copy()
test_grouped = data[data["group_id"].isin(test_groups)].copy()

print("Transaktionen:")
print("Training:", len(train_grouped))
print("Validation:", len(val_grouped))
print("Test:", len(test_grouped))

print("\nKategorien:")
print("Training:", train_grouped["categoryName"].nunique())
print("Validation:", val_grouped["categoryName"].nunique())
print("Test:", test_grouped["categoryName"].nunique())

print("\nGruppen:")
print("Training:", train_grouped["group_id"].nunique())
print("Validation:", val_grouped["group_id"].nunique())
print("Test:", test_grouped["group_id"].nunique())

print("\nÜberschneidungen:")
print("Train/Val:", len(set(train_groups) & set(val_groups)))
print("Train/Test:", len(set(train_groups) & set(test_groups)))
print("Val/Test:", len(set(val_groups) & set(test_groups)))

In [ ]:
# Größe jeder einzelnen Gruppe bestimmen
group_sizes = (
    data.groupby(["categoryName", "group_id"])
    .size()
    .reset_index(name="size")
)

train_groups = []
val_groups = []
test_groups = []

rng = np.random.RandomState(42)

for category, cat_groups in group_sizes.groupby("categoryName"):

    # kleine Gruppen zuerst, damit Validation/Test
    # nicht unnötig riesig werden
    cat_groups = cat_groups.sort_values("size")

    ids = cat_groups["group_id"].tolist()
    n = len(ids)

    if n == 1:
        # einzige verfügbare Gruppe muss ins Training
        train_groups.append(ids[0])

    elif n == 2:
        # eine Gruppe Training, eine Test
        test_groups.append(ids[0])
        train_groups.append(ids[1])

    else:
        # mindestens drei Gruppen:
        # kleinste -> Validation
        # zweitkleinste -> Test
        # Rest -> Training
        val_groups.append(ids[0])
        test_groups.append(ids[1])
        train_groups.extend(ids[2:])


train_grouped = data[data["group_id"].isin(train_groups)].copy()
val_grouped = data[data["group_id"].isin(val_groups)].copy()
test_grouped = data[data["group_id"].isin(test_groups)].copy()

print("Transaktionen:")
print("Training:", len(train_grouped))
print("Validation:", len(val_grouped))
print("Test:", len(test_grouped))

print("\nKategorien:")
print("Training:", train_grouped["categoryName"].nunique())
print("Validation:", val_grouped["categoryName"].nunique())
print("Test:", test_grouped["categoryName"].nunique())

print("\nGruppen:")
print("Training:", train_grouped["group_id"].nunique())
print("Validation:", val_grouped["group_id"].nunique())
print("Test:", test_grouped["group_id"].nunique())

print("\nÜberschneidungen:")
print("Train/Val:", len(set(train_groups) & set(val_groups)))
print("Train/Test:", len(set(train_groups) & set(test_groups)))
print("Val/Test:", len(set(val_groups) & set(test_groups)))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

paraphrase_model_name = "seduerr/t5_base_paws_ger"

para_tokenizer = AutoTokenizer.from_pretrained(paraphrase_model_name)
para_model = AutoModelForSeq2SeqLM.from_pretrained(paraphrase_model_name)

para_model.to(device)

print("Deutsches Paraphrasierungsmodell erfolgreich geladen.")

In [ ]:
def generate_paraphrases(text, num_return_sequences=3):
    prompt = f"paraphrase: {text}"

    inputs = para_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    outputs = para_model.generate(
        **inputs,
        max_length=128,
        num_beams=6,
        num_return_sequences=num_return_sequences,
        do_sample=False
    )

    return [
        para_tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]


test_texts = [
    "software lizenz",
    "gehaltslauf",
    "consulting retainer",
    "domain und server",
    "google ads budget"
]

for text in test_texts:
    print("\nORIGINAL:", text)

    paraphrases = generate_paraphrases(text)

    for i, p in enumerate(paraphrases, 1):
        print(f"{i}: {p}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

generation_model_name = "google/flan-t5-small"

gen_tokenizer = AutoTokenizer.from_pretrained(generation_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(generation_model_name)

gen_model.to(device)

print("FLAN-T5 erfolgreich geladen.")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gen_model.to(device)

print("Gerät:", device)
print("FLAN-T5 ist bereit.")

In [ ]:
def generate_flan_variants(text, category, num_return_sequences=3):
    prompt = (
        f"Erzeuge {num_return_sequences} kurze alternative "
        f"Banktransaktionsbeschreibungen auf Deutsch für die Kategorie "
        f"'{category}'. Ausgangstext: '{text}'. "
        f"Behalte die Bedeutung bei. Gib nur die Varianten aus."
    )

    inputs = gen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    outputs = gen_model.generate(
        **inputs,
        max_length=64,
        num_beams=6,
        num_return_sequences=num_return_sequences,
        do_sample=False
    )

    return [
        gen_tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]


test_examples = [
    ("software lizenz", "IT & Software"),
    ("gehaltslauf", "Löhne & Gehälter"),
    ("consulting retainer", "Dienstleistungen"),
    ("domain und server", "IT & Software"),
    ("google ads budget", "Marketing & Vertrieb")
]

for text, category in test_examples:
    print("\nORIGINAL:", text)
    print("KATEGORIE:", category)

    variants = generate_flan_variants(text, category)

    for i, variant in enumerate(variants, 1):
        print(f"{i}: {variant}")

In [ ]:
import random
import pandas as pd

random.seed(42)

# einfache realistische Ergänzungen für kurze Transaktionsbeschreibungen
prefixes = [
    "zahlung",
    "rechnung",
    "monatlich",
    "jährlich",
    "gebühr",
    "kosten",
    "buchung",
    "abbuchung"
]

suffixes = [
    "monatlich",
    "jährlich",
    "rechnung",
    "zahlung",
    "kosten",
    "gebühr",
    "service",
    "vertrag"
]

def augment_text(text):
    text = str(text).strip()

    methods = [
        lambda x: f"{random.choice(prefixes)} {x}",
        lambda x: f"{x} {random.choice(suffixes)}",
        lambda x: " ".join(reversed(x.split())) if len(x.split()) > 1 else x,
    ]

    new_text = random.choice(methods)(text)
    return clean_description(new_text)


# Ziel: seltene Kategorien stärker auffüllen
category_counts = train_grouped["categoryName"].value_counts()

# nicht komplett auf 240 angleichen -> moderates Ziel
target_count = 80

synthetic_rows = []

for category, count in category_counts.items():

    if count >= target_count:
        continue

    needed = target_count - count

    category_data = train_grouped[
        train_grouped["categoryName"] == category
    ]

    for _ in range(needed):

        source_row = category_data.sample(
            n=1,
            random_state=random.randint(0, 100000)
        ).iloc[0]

        new_description = augment_text(
            source_row["clean_description"]
        )

        synthetic_rows.append({
            "description": new_description,
            "clean_description": new_description,
            "categoryName": category,
            "parentCategoryName": source_row["parentCategoryName"],
            "synthetic": True
        })


synthetic_data = pd.DataFrame(synthetic_rows)

print("Erzeugte synthetische Beispiele:", len(synthetic_data))

display(
    synthetic_data[
        ["description", "categoryName"]
    ].head(30)
)

In [ ]:
# Verbesserte regelbasierte Augmentation

prefixes = [
    "zahlung",
    "rechnung",
    "buchung",
    "abbuchung"
]

suffixes = [
    "monatlich",
    "jährlich",
    "zahlung",
    "rechnung",
    "kosten",
    "gebühr"
]

def augment_text_v2(text):
    text = str(text).strip()

    if random.random() < 0.5:
        new_text = f"{random.choice(prefixes)} {text}"
    else:
        new_text = f"{text} {random.choice(suffixes)}"

    return clean_description(new_text)


synthetic_rows_v2 = []

# Moderateres Ziel
target_count = 60

category_counts = train_grouped["categoryName"].value_counts()

for category, count in category_counts.items():

    if count >= target_count:
        continue

    needed = target_count - count

    category_data = train_grouped[
        train_grouped["categoryName"] == category
    ]

    for _ in range(needed * 3):  # mehr Versuche wegen späterer Duplikatentfernung

        source_row = category_data.sample(
            n=1,
            random_state=random.randint(0, 100000)
        ).iloc[0]

        new_description = augment_text_v2(
            source_row["clean_description"]
        )

        synthetic_rows_v2.append({
            "description": new_description,
            "clean_description": new_description,
            "categoryName": category,
            "parentCategoryName": source_row["parentCategoryName"],
            "synthetic": True
        })


synthetic_data_v2 = pd.DataFrame(synthetic_rows_v2)

# Doppelte synthetische Texte innerhalb derselben Kategorie entfernen
synthetic_data_v2 = synthetic_data_v2.drop_duplicates(
    subset=["clean_description", "categoryName"]
)

# Texte entfernen, die bereits im echten Training vorkommen
real_pairs = set(
    zip(
        train_grouped["clean_description"],
        train_grouped["categoryName"]
    )
)

synthetic_data_v2 = synthetic_data_v2[
    ~synthetic_data_v2.apply(
        lambda row: (
            row["clean_description"],
            row["categoryName"]
        ) in real_pairs,
        axis=1
    )
]

print("Eindeutige synthetische Beispiele:", len(synthetic_data_v2))

display(
    synthetic_data_v2[
        ["description", "categoryName"]
    ].head(30)
)

In [ ]:
# Konservative regelbasierte Datenaugmentation

transaction_terms = [
    "zahlung",
    "rechnung",
    "buchung",
    "abbuchung"
]

def augment_text_v3(text):
    text = str(text).strip()

    if random.random() < 0.5:
        new_text = f"{random.choice(transaction_terms)} {text}"
    else:
        new_text = f"{text} {random.choice(transaction_terms)}"

    return clean_description(new_text)


synthetic_rows_v3 = []

# Seltene Klassen moderat ergänzen
target_count = 60

category_counts = train_grouped["categoryName"].value_counts()

for category, count in category_counts.items():

    if count >= target_count:
        continue

    category_data = train_grouped[
        train_grouped["categoryName"] == category
    ]

    # Mehr Versuche erzeugen, anschließend Duplikate entfernen
    for _ in range((target_count - count) * 4):

        source_row = category_data.sample(
            n=1,
            random_state=random.randint(0, 100000)
        ).iloc[0]

        new_description = augment_text_v3(
            source_row["clean_description"]
        )

        synthetic_rows_v3.append({
            "description": new_description,
            "clean_description": new_description,
            "categoryName": category,
            "parentCategoryName": source_row["parentCategoryName"],
            "synthetic": True
        })


synthetic_data_v3 = pd.DataFrame(synthetic_rows_v3)

# Doppelte synthetische Beispiele entfernen
synthetic_data_v3 = synthetic_data_v3.drop_duplicates(
    subset=["clean_description", "categoryName"]
)

# Bereits im echten Trainingsdatensatz vorhandene Texte entfernen
real_pairs = set(
    zip(
        train_grouped["clean_description"],
        train_grouped["categoryName"]
    )
)

synthetic_data_v3 = synthetic_data_v3[
    ~synthetic_data_v3.apply(
        lambda row: (
            row["clean_description"],
            row["categoryName"]
        ) in real_pairs,
        axis=1
    )
]

print("Eindeutige synthetische Beispiele:", len(synthetic_data_v3))

display(
    synthetic_data_v3[
        ["description", "categoryName"]
    ].head(30)
)

In [ ]:
# Markierung für echte Trainingsdaten
train_real = train_grouped.copy()
train_real["synthetic"] = False

# Relevante Spalten angleichen
columns = [
    "description",
    "clean_description",
    "categoryName",
    "parentCategoryName",
    "synthetic"
]

train_real = train_real[columns]
synthetic_final = synthetic_data_v3[columns]

# Echtes + synthetisches Training zusammenführen
train_augmented = pd.concat(
    [train_real, synthetic_final],
    ignore_index=True
)

print("Echte Trainingsdaten:", len(train_real))
print("Synthetische Trainingsdaten:", len(synthetic_final))
print("Gesamtes Trainingsset:", len(train_augmented))

print("\nKlassenverteilung nach Augmentation:")
display(
    train_augmented["categoryName"]
    .value_counts()
    .sort_values(ascending=False)
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder_v2 = LabelEncoder()

# Encoder kennt alle 37 Kategorien aus dem Training
label_encoder_v2.fit(train_augmented["categoryName"])

train_augmented["label"] = label_encoder_v2.transform(
    train_augmented["categoryName"]
)

val_grouped["label"] = label_encoder_v2.transform(
    val_grouped["categoryName"]
)

test_grouped["label"] = label_encoder_v2.transform(
    test_grouped["categoryName"]
)

print("Anzahl Klassen:", len(label_encoder_v2.classes_))
print("Training:", len(train_augmented))
print("Validation:", len(val_grouped))
print("Test:", len(test_grouped))

In [ ]:
# Neue Daten mit den bereinigten Beschreibungen tokenisieren

train_encodings_v2 = tokenizer(
    train_augmented["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings_v2 = tokenizer(
    val_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings_v2 = tokenizer(
    test_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("Tokenisierung abgeschlossen.")
print("Training:", len(train_encodings_v2["input_ids"]))
print("Validation:", len(val_encodings_v2["input_ids"]))
print("Test:", len(test_encodings_v2["input_ids"]))

In [ ]:
train_dataset_v2 = TransactionDataset(
    train_encodings_v2,
    train_augmented["label"].tolist()
)

val_dataset_v2 = TransactionDataset(
    val_encodings_v2,
    val_grouped["label"].tolist()
)

test_dataset_v2 = TransactionDataset(
    test_encodings_v2,
    test_grouped["label"].tolist()
)

print("Neue PyTorch-Datensätze erstellt.")
print("Training:", len(train_dataset_v2))
print("Validation:", len(val_dataset_v2))
print("Test:", len(test_dataset_v2))

In [ ]:
from transformers import AutoModelForSequenceClassification

model_v2 = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=37,
    ignore_mismatched_sizes=True
)

model_v2.to(device)

print("Frisches FinBERT-Modell geladen.")

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Neue Klassengewichte auf Basis des augmentierten Trainingssets
classes_v2 = np.arange(len(label_encoder_v2.classes_))

class_weights_v2 = compute_class_weight(
    class_weight="balanced",
    classes=classes_v2,
    y=train_augmented["label"].to_numpy()
)

class_weights_v2 = torch.tensor(
    class_weights_v2,
    dtype=torch.float
).to(device)

criterion_v2 = FocalLoss(
    alpha=class_weights_v2,
    gamma=2.0
)

# DataLoader
batch_size = 16
epochs = 3
learning_rate = 2e-5

train_loader_v2 = DataLoader(
    train_dataset_v2,
    batch_size=batch_size,
    shuffle=True
)

val_loader_v2 = DataLoader(
    val_dataset_v2,
    batch_size=batch_size,
    shuffle=False
)

# Optimizer
optimizer_v2 = AdamW(
    model_v2.parameters(),
    lr=learning_rate
)

# Scheduler mit 500 Warmup Steps 
total_training_steps = len(train_loader_v2) * epochs

scheduler_v2 = get_linear_schedule_with_warmup(
    optimizer_v2,
    num_warmup_steps=500,
    num_training_steps=total_training_steps
)

print("Weighted Focal Loss vorbereitet.")
print("Training Steps gesamt:", total_training_steps)
print("Warmup Steps:", 500)

In [ ]:
# Scheduler mit angepasstem Warmup
total_training_steps = len(train_loader_v2) * epochs

warmup_steps = int(0.10 * total_training_steps)

scheduler_v2 = get_linear_schedule_with_warmup(
    optimizer_v2,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Weighted Focal Loss vorbereitet.")
print("Training Steps gesamt:", total_training_steps)
print("Warmup Steps:", warmup_steps)

In [ ]:
from tqdm.auto import tqdm

for epoch in range(epochs):

    print(f"\n===== Epoche {epoch + 1}/{epochs} =====")

    # TRAINING
    model_v2.train()
    total_train_loss = 0

    progress_bar = tqdm(
        train_loader_v2,
        desc="Training"
    )

    for batch in progress_bar:

        optimizer_v2.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_v2(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer_v2.step()
        scheduler_v2.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = (
        total_train_loss / len(train_loader_v2)
    )

    # VALIDATION
    model_v2.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_v2:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model_v2(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_v2(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = (
        total_val_loss / len(val_loader_v2)
    )

    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
from sklearn.metrics import classification_report
from collections import Counter

model_v2.eval()

val_predictions = []
val_true_labels = []

with torch.no_grad():

    for batch in val_loader_v2:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_v2(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        val_predictions.extend(
            predictions.cpu().numpy()
        )

        val_true_labels.extend(
            labels.cpu().numpy()
        )


predicted_names = label_encoder_v2.inverse_transform(
    val_predictions
)

true_names = label_encoder_v2.inverse_transform(
    val_true_labels
)

print("Am häufigsten vorhergesagte Kategorien:")
print(Counter(predicted_names).most_common(15))

print("\nTatsächliche Kategorien:")
print(Counter(true_names).most_common(15))

In [ ]:
val_results = val_grouped[
    ["clean_description", "categoryName"]
].copy()

val_results["prediction"] = predicted_names
val_results["correct"] = (
    val_results["categoryName"] == val_results["prediction"]
)

display(
    val_results[
        ~val_results["correct"]
    ][
        ["clean_description", "categoryName", "prediction"]
    ].head(50)
)

In [ ]:
problem_categories = [
    "IT & Software",
    "Marketing & Vertrieb",
    "Energie",
    "Miete & Nebenkosten"
]

for category in problem_categories:
    print("\n==============================")
    print("KATEGORIE:", category)
    print("==============================")

    examples = (
        train_grouped[
            train_grouped["categoryName"] == category
        ]["clean_description"]
        .drop_duplicates()
        .tolist()
    )

    for text in examples:
        print("-", text)

    print("Anzahl Grundmuster:", len(examples))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalLossV2(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        # Wahrscheinlichkeiten für alle 37 Klassen
        probs = F.softmax(logits, dim=1)

        # Wahrscheinlichkeit der tatsächlich richtigen Klasse
        pt = probs.gather(
            1,
            targets.unsqueeze(1)
        ).squeeze(1)

        # numerische Stabilität
        pt = pt.clamp(min=1e-8, max=1.0)

        # Klassengewicht der richtigen Klasse
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
        else:
            alpha_t = 1.0

        # Focal-Loss-Formel
        loss = (
            -alpha_t
            * (1 - pt) ** self.gamma
            * torch.log(pt)
        )

        return loss.mean()


criterion_v3 = FocalLossV2(
    alpha=class_weights_v2,
    gamma=2.0
)

print("Korrigierte Focal Loss vorbereitet.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# TF-IDF : Uni- und Bigramme
tfidf_baseline = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

# Nur auf Trainingsdaten fitten
X_train_tfidf = tfidf_baseline.fit_transform(
    train_augmented["clean_description"]
)

X_val_tfidf = tfidf_baseline.transform(
    val_grouped["clean_description"]
)

# Logistic Regression Baseline
logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logreg.fit(
    X_train_tfidf,
    train_augmented["label"]
)

val_pred_logreg = logreg.predict(X_val_tfidf)

val_acc_logreg = accuracy_score(
    val_grouped["label"],
    val_pred_logreg
)

val_f1_logreg = f1_score(
    val_grouped["label"],
    val_pred_logreg,
    average="macro",
    zero_division=0
)

print(f"Validation Accuracy: {val_acc_logreg:.4f}")
print(f"Validation Macro-F1: {val_f1_logreg:.4f}")

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        val_grouped["label"],
        val_pred_logreg,
        labels=sorted(val_grouped["label"].unique()),
        target_names=label_encoder_v2.inverse_transform(
            sorted(val_grouped["label"].unique())
        ),
        zero_division=0
    )
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model_v3 = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=37,
    ignore_mismatched_sizes=True
)

model_v3.to(device)

print("Frisches FinBERT V3 geladen.")

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Trainingsparameter
batch_size = 16
epochs = 3
learning_rate = 2e-5

# neuer Optimizer für model_v3
optimizer_v3 = AdamW(
    model_v3.parameters(),
    lr=learning_rate
)

# Gesamtanzahl der Trainingsschritte
total_training_steps_v3 = len(train_loader_v2) * epochs

# 10 % Warmup
warmup_steps_v3 = int(0.10 * total_training_steps_v3)

# neuer Scheduler
scheduler_v3 = get_linear_schedule_with_warmup(
    optimizer_v3,
    num_warmup_steps=warmup_steps_v3,
    num_training_steps=total_training_steps_v3
)

print("Optimizer und Scheduler vorbereitet.")
print("Training Steps gesamt:", total_training_steps_v3)
print("Warmup Steps:", warmup_steps_v3)

In [ ]:
from tqdm.auto import tqdm

for epoch in range(epochs):

    print(f"\n===== Epoche {epoch + 1}/{epochs} =====")

    # TRAINING
    model_v3.train()
    total_train_loss = 0

    progress_bar = tqdm(
        train_loader_v2,
        desc="Training"
    )

    for batch in progress_bar:

        optimizer_v3.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_v3(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_v3(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer_v3.step()
        scheduler_v3.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = (
        total_train_loss / len(train_loader_v2)
    )

    # VALIDATION
    model_v3.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_v2:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model_v3(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_v3(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = (
        total_val_loss / len(val_loader_v2)
    )

    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Frisches Modell
model_v4 = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=37,
    ignore_mismatched_sizes=True
)

model_v4.to(device)

# Neuer Optimizer
optimizer_v4 = AdamW(
    model_v4.parameters(),
    lr=2e-5
)

# Diesmal genau 1 Epoche
epochs_v4 = 1

total_steps_v4 = len(train_loader_v2) * epochs_v4
warmup_steps_v4 = int(0.10 * total_steps_v4)

scheduler_v4 = get_linear_schedule_with_warmup(
    optimizer_v4,
    num_warmup_steps=warmup_steps_v4,
    num_training_steps=total_steps_v4
)

print("FinBERT V4 vorbereitet.")
print("Training Steps:", total_steps_v4)
print("Warmup Steps:", warmup_steps_v4)

In [ ]:
from tqdm.auto import tqdm

model_v4.train()
total_train_loss_v4 = 0

progress_bar = tqdm(
    train_loader_v2,
    desc="Training V4"
)

for batch in progress_bar:

    optimizer_v4.zero_grad()

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model_v4(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    loss = criterion_v3(
        outputs.logits,
        labels
    )

    loss.backward()

    optimizer_v4.step()
    scheduler_v4.step()

    total_train_loss_v4 += loss.item()

    progress_bar.set_postfix(
        loss=f"{loss.item():.4f}"
    )

avg_train_loss_v4 = (
    total_train_loss_v4 / len(train_loader_v2)
)

print(f"Train Loss V4: {avg_train_loss_v4:.4f}")

In [ ]:
model_v4.eval()

correct = 0
total = 0
total_val_loss_v4 = 0

with torch.no_grad():

    for batch in val_loader_v2:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_v4(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_v3(
            outputs.logits,
            labels
        )

        total_val_loss_v4 += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

avg_val_loss_v4 = (
    total_val_loss_v4 / len(val_loader_v2)
)

val_accuracy_v4 = correct / total

print(f"Validation Loss V4: {avg_val_loss_v4:.4f}")
print(f"Validation Accuracy V4: {val_accuracy_v4:.4f}")

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print("Seed gesetzt:", seed)

In [ ]:
from copy import deepcopy
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

# Frisches Modell
model_final = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=37,
    ignore_mismatched_sizes=True
)

model_final.to(device)

# Optimizer
optimizer_final = AdamW(
    model_final.parameters(),
    lr=2e-5
)

epochs_final = 3

total_steps_final = len(train_loader_v2) * epochs_final
warmup_steps_final = int(0.10 * total_steps_final)

scheduler_final = get_linear_schedule_with_warmup(
    optimizer_final,
    num_warmup_steps=warmup_steps_final,
    num_training_steps=total_steps_final
)

best_val_accuracy = 0.0
best_model_state = None
best_epoch = None

for epoch in range(epochs_final):

    print(f"\n===== Epoche {epoch + 1}/{epochs_final} =====")

    # TRAINING
    model_final.train()
    total_train_loss = 0

    progress_bar = tqdm(
        train_loader_v2,
        desc="Training"
    )

    for batch in progress_bar:

        optimizer_final.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_final(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_v3(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer_final.step()
        scheduler_final.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader_v2)

    # VALIDATION
    model_final.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_v2:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model_final(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_v3(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader_v2)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    # Besten Modellzustand speichern
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1
        best_model_state = deepcopy(
            model_final.state_dict()
        )

        print("-> Neuer bester Checkpoint gespeichert.")


# Besten Checkpoint zurückladen
model_final.load_state_dict(best_model_state)

print("\nBestes Modell geladen.")
print("Beste Epoche:", best_epoch)
print("Beste Validation Accuracy:", round(best_val_accuracy, 4))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

german_model_name = "scherrmann/GermanFinBert_FP"

german_tokenizer = AutoTokenizer.from_pretrained(german_model_name)

german_model = AutoModelForSequenceClassification.from_pretrained(
    german_model_name,
    num_labels=37,
    ignore_mismatched_sizes=True
)

german_model.to(device)

print("German FinBERT erfolgreich geladen.")

In [ ]:
# Daten mit dem deutschen Tokenizer neu tokenisieren

train_encodings_de = german_tokenizer(
    train_augmented["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings_de = german_tokenizer(
    val_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings_de = german_tokenizer(
    test_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("Deutsche Tokenisierung abgeschlossen.")
print("Training:", len(train_encodings_de["input_ids"]))
print("Validation:", len(val_encodings_de["input_ids"]))
print("Test:", len(test_encodings_de["input_ids"]))

In [ ]:
train_dataset_de = TransactionDataset(
    train_encodings_de,
    train_augmented["label"].tolist()
)

val_dataset_de = TransactionDataset(
    val_encodings_de,
    val_grouped["label"].tolist()
)

test_dataset_de = TransactionDataset(
    test_encodings_de,
    test_grouped["label"].tolist()
)

print("GermanFinBERT-Datensätze erstellt.")
print("Training:", len(train_dataset_de))
print("Validation:", len(val_dataset_de))
print("Test:", len(test_dataset_de))

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# DataLoader für GermanFinBERT
train_loader_de = DataLoader(
    train_dataset_de,
    batch_size=16,
    shuffle=True
)

val_loader_de = DataLoader(
    val_dataset_de,
    batch_size=16,
    shuffle=False
)

test_loader_de = DataLoader(
    test_dataset_de,
    batch_size=16,
    shuffle=False
)

# Optimizer
optimizer_de = AdamW(
    german_model.parameters(),
    lr=2e-5
)

epochs_de = 3

total_steps_de = len(train_loader_de) * epochs_de
warmup_steps_de = int(0.10 * total_steps_de)

scheduler_de = get_linear_schedule_with_warmup(
    optimizer_de,
    num_warmup_steps=warmup_steps_de,
    num_training_steps=total_steps_de
)

print("GermanFinBERT Training vorbereitet.")
print("Batches pro Epoche:", len(train_loader_de))
print("Training Steps gesamt:", total_steps_de)
print("Warmup Steps:", warmup_steps_de)

In [ ]:
from copy import deepcopy
from tqdm.auto import tqdm

best_val_accuracy_de = 0.0
best_model_state_de = None
best_epoch_de = None

for epoch in range(epochs_de):

    print(f"\n===== GermanFinBERT Epoche {epoch + 1}/{epochs_de} =====")

    # TRAINING
    german_model.train()
    total_train_loss = 0

    progress_bar = tqdm(
        train_loader_de,
        desc="Training GermanFinBERT"
    )

    for batch in progress_bar:

        optimizer_de.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = german_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_v3(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer_de.step()
        scheduler_de.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = total_train_loss / len(train_loader_de)

    # VALIDATION
    german_model.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_de:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = german_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_v3(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader_de)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    # besten Checkpoint speichern
    if val_accuracy > best_val_accuracy_de:

        best_val_accuracy_de = val_accuracy
        best_epoch_de = epoch + 1

        best_model_state_de = deepcopy(
            german_model.state_dict()
        )

        print("-> Neuer bester Checkpoint gespeichert.")


# besten Checkpoint wieder laden
german_model.load_state_dict(best_model_state_de)

print("\nBestes GermanFinBERT-Modell geladen.")
print("Beste Epoche:", best_epoch_de)
print("Beste Validation Accuracy:", round(best_val_accuracy_de, 4))

In [ ]:
import pandas as pd

# ============================================================
# 1. ÜBERSICHT
# ============================================================

overview = pd.DataFrame({
    "Kennzahl": [
        "Gesamte Transaktionen",
        "Oberkategorien",
        "Unterkategorien",
        "Eindeutige Gruppen / Grundmuster",
        "Training vor Augmentation",
        "Synthetische Trainingsbeispiele",
        "Training nach Augmentation",
        "Validation",
        "Test",
        "Train/Validation Gruppenüberschneidung",
        "Train/Test Gruppenüberschneidung",
        "Validation/Test Gruppenüberschneidung"
    ],
    "Wert": [
        2000,
        11,
        37,
        170,
        len(train_real),
        len(synthetic_final),
        len(train_augmented),
        len(val_grouped),
        len(test_grouped),
        0,
        0,
        0
    ]
})


# ============================================================
# 2. MODELLERGEBNISSE
# ============================================================

model_results = pd.DataFrame({
    "Modell / Experiment": [
        "FinBERT – ursprünglicher Random Split",
        "FinBERT V2 – Group Split",
        "FinBERT V3 – Group Split, korrigierte Focal Loss",
        "TF-IDF + Logistic Regression – Group Split",
        "FinBERT Final – Group Split, Seed 42",
        "GermanFinBERT – Group Split"
    ],
    "Validation Accuracy": [
        None,
        0.0840,
        0.4809,
        0.0992,
        0.0916,
        0.0992
    ],
    "Test Accuracy": [
        1.0000,
        None,
        None,
        None,
        None,
        None
    ],
    "Hinweis": [
        "Nicht aussagekräftig: Leakage durch identische/ähnliche Transaktionsmuster",
        "Sauberer Group Split; keine Gruppenüberschneidung",
        "Bester beobachteter Lauf; Ergebnis stark von Initialisierung abhängig",
        "Baseline auf identischem Group Split",
        "Reproduzierbarer Lauf mit festem Seed 42",
        "Deutsches Finanz-BERT; bester Checkpoint Epoche 2"
    ]
})


# ============================================================
# 3. KATEGORIEN UND GRUNDMUSTER
# ============================================================

transaction_counts = (
    train_grouped
    .groupby("categoryName")
    .size()
    .rename("Training_Transaktionen")
)

group_counts = (
    train_grouped
    .groupby("categoryName")["group_id"]
    .nunique()
    .rename("Training_Grundmuster")
)

category_summary = pd.concat(
    [transaction_counts, group_counts],
    axis=1
).reset_index()

category_summary = category_summary.sort_values(
    "Training_Grundmuster"
)


# ============================================================
# 4. FEHLKLASSIFIKATIONEN
# ============================================================

# GermanFinBERT auf Validation vorhersagen
german_model.eval()

val_predictions_de = []
val_true_de = []

with torch.no_grad():

    for batch in val_loader_de:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = german_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        val_predictions_de.extend(
            predictions.cpu().numpy()
        )

        val_true_de.extend(
            labels.cpu().numpy()
        )


prediction_names_de = label_encoder_v2.inverse_transform(
    val_predictions_de
)

validation_results = val_grouped[
    ["clean_description", "categoryName"]
].copy()

validation_results["Vorhersage"] = prediction_names_de

validation_results["Korrekt"] = (
    validation_results["categoryName"]
    == validation_results["Vorhersage"]
)

errors = validation_results[
    ~validation_results["Korrekt"]
].copy()

errors.columns = [
    "Transaktionsbeschreibung",
    "Tatsächliche Kategorie",
    "Vorhersage",
    "Korrekt"
]


# ============================================================
# EXCEL EXPORT
# ============================================================

filename = "Ergebnisuebersicht_FinBERT.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    overview.to_excel(
        writer,
        sheet_name="Übersicht",
        index=False
    )

    model_results.to_excel(
        writer,
        sheet_name="Modellergebnisse",
        index=False
    )

    category_summary.to_excel(
        writer,
        sheet_name="Kategorien",
        index=False
    )

    errors.to_excel(
        writer,
        sheet_name="Fehlklassifikationen",
        index=False
    )


print("Excel-Datei erfolgreich erstellt:")
print(filename)

# Finaler Versuchsaufbau – LLM-basierte Datenaugmentation

In [ ]:
# ============================================================
# AUSGANGSLAGE DES FINALEN EXPERIMENTS
# ============================================================

print("Training:", len(train_grouped))
print("Validation:", len(val_grouped))
print("Test:", len(test_grouped))

print("\nAnzahl Kategorien:")
print(train_grouped["categoryName"].nunique())

# Anzahl realer Transaktionen pro Kategorie
transaction_counts = (
    train_grouped
    .groupby("categoryName")
    .size()
    .sort_values()
)

# Anzahl unterschiedlicher Grundmuster pro Kategorie
group_counts = (
    train_grouped
    .groupby("categoryName")["group_id"]
    .nunique()
    .sort_values()
)

category_distribution = (
    transaction_counts
    .rename("Transaktionen")
    .to_frame()
    .join(group_counts.rename("Grundmuster"))
)

display(category_distribution)

In [ ]:
# ============================================================
# AUGMENTATIONSPLAN
# Ziel: mindestens 15 unterschiedliche Grundmuster pro Kategorie
# ============================================================

TARGET_PATTERNS = 15

augmentation_plan = (
    category_distribution
    .copy()
)

augmentation_plan["synthetic_needed"] = (
    TARGET_PATTERNS - augmentation_plan["Grundmuster"]
).clip(lower=0)

augmentation_plan = augmentation_plan.sort_values(
    "synthetic_needed",
    ascending=False
)

display(augmentation_plan)

print(
    "\nGeplante synthetische Beispiele insgesamt:",
    int(augmentation_plan["synthetic_needed"].sum())
)

In [ ]:
%pip install openai

In [ ]:
import os
import getpass

os.environ["AZURE_OPENAI_ENDPOINT"] = input(
    "Azure OpenAI Endpoint: "
).strip()

os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Azure OpenAI API Key: ")

os.environ["AZURE_OPENAI_DEPLOYMENT"] = input(
    "Azure OpenAI Deployment Name: "
).strip()

os.environ["AZURE_OPENAI_API_VERSION"] = input(
    "Azure OpenAI API Version: "
).strip()

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = input("Azure OpenAI Endpoint: ").strip()
os.environ["AZURE_OPENAI_DEPLOYMENT"] = input("Azure OpenAI Deployment Name: ").strip()
os.environ["AZURE_OPENAI_API_VERSION"] = "2024-12-01-preview"

In [ ]:
import getpass
import os

os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Azure OpenAI API Key: ")

In [ ]:
from openai import AzureOpenAI
import os

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"]
)

response = client.chat.completions.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    messages=[
        {
            "role": "user",
            "content": "Antworte ausschließlich mit TEST."
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
import os

print(bool(os.environ.get("AZURE_OPENAI_API_KEY")))

In [ ]:
import os



os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Azure OpenAI API Key: ")

In [ ]:
print(bool(os.environ.get("AZURE_OPENAI_API_KEY")))

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/openai/v1/"
)

response = client.chat.completions.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    messages=[
        {
            "role": "user",
            "content": "Antworte ausschließlich mit TEST."
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
import json

test_category = "Boni / Prämien"

real_examples = (
    train_grouped.loc[
        train_grouped["categoryName"] == test_category,
        "clean_description"
    ]
    .drop_duplicates()
    .tolist()
)

print("Kategorie:", test_category)
print("Reale Grundmuster:", real_examples)

In [ ]:
import json

def generate_synthetic_transactions(category, examples, n):

    prompt = f"""
Du erzeugst realistische kurze Banktransaktionsbeschreibungen
für Trainingsdaten eines Klassifikationsmodells für KMU-Finanztransaktionen.

Kategorie:
{category}

Vorhandene reale Beispiele:
{json.dumps(examples, ensure_ascii=False)}

Erzeuge genau {n} neue Transaktionsbeschreibungen.

Anforderungen:
- alle Beispiele müssen eindeutig zur angegebenen Kategorie gehören
- nicht einfach vorhandene Beispiele leicht umformulieren
- unterschiedliche realistische Situationen verwenden
- kurz und banktypisch formulieren
- keine Erklärungen
- keine Nummerierung
- keine zusätzlichen Kategorien
- Erzeuge keine Beispiele, die auch plausibel zu einer anderen Finanzkategorie
  wie Rückvergütungen, Provisionen, Gehälter oder sonstigen Zahlungen gehören könnten.
- gib ausschließlich ein JSON-Array mit genau {n} Strings zurück
"""

    response = client.chat.completions.create(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_text = response.choices[0].message.content.strip()

    synthetic = json.loads(raw_text)

    if not isinstance(synthetic, list):
        raise ValueError("Antwort ist keine Liste.")

    if len(synthetic) != n:
        raise ValueError(
            f"Erwartet: {n} Beispiele, erhalten: {len(synthetic)}"
        )

    if not all(isinstance(x, str) for x in synthetic):
        raise ValueError("Nicht alle Elemente sind Strings.")

    return synthetic

In [ ]:
test_output = generate_synthetic_transactions(
    category="Boni / Prämien",
    examples=["bonuszahlung mitarbeitende"],
    n=5
)

test_output

In [ ]:
import json

all_categories = sorted(
    train_grouped["categoryName"]
    .dropna()
    .unique()
    .tolist()
)

def generate_synthetic_transactions(category, examples, n):

    prompt = f"""
Du erzeugst realistische kurze Banktransaktionsbeschreibungen
für Trainingsdaten eines Klassifikationsmodells für KMU-Finanztransaktionen.

Zielkategorie:
{category}

Alle möglichen Kategorien im Datensatz:
{json.dumps(all_categories, ensure_ascii=False)}

Vorhandene reale Beispiele der Zielkategorie:
{json.dumps(examples, ensure_ascii=False)}

Erzeuge genau {n} neue Transaktionsbeschreibungen.

Anforderungen:
- alle Beispiele müssen eindeutig zur Zielkategorie gehören
- kein Beispiel darf fachlich besser zu einer anderen vorhandenen Kategorie passen
- vorhandene Beispiele nicht kopieren
- nicht nur einzelne Wörter austauschen
- unterschiedliche realistische Situationen und Formulierungen verwenden
- kurz und banktypisch formulieren
- keine Erklärungen
- keine Nummerierung
- keine zusätzlichen Kategorien
- ausschließlich ein JSON-Array mit genau {n} Strings zurückgeben
"""

    response = client.chat.completions.create(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_text = response.choices[0].message.content.strip()

    synthetic = json.loads(raw_text)

    if not isinstance(synthetic, list):
        raise ValueError("Antwort ist keine Liste.")

    if len(synthetic) != n:
        raise ValueError(
            f"Erwartet: {n} Beispiele, erhalten: {len(synthetic)}"
        )

    if not all(isinstance(x, str) for x in synthetic):
        raise ValueError("Nicht alle Elemente sind Strings.")

    return synthetic

In [ ]:
import pandas as pd

synthetic_rows = []

for category, row in augmentation_plan.iterrows():

    n = int(row["synthetic_needed"])

    if n == 0:
        continue

    examples = (
        train_grouped.loc[
            train_grouped["categoryName"] == category,
            "clean_description"
        ]
        .drop_duplicates()
        .tolist()
    )

    print(f"Generiere {category}: {n} Beispiele ...")

    generated = generate_synthetic_transactions(
        category=category,
        examples=examples,
        n=n
    )

    for text in generated:
        synthetic_rows.append({
            "clean_description": text,
            "categoryName": category
        })

    # nach jeder Kategorie sichern
    pd.DataFrame(synthetic_rows).to_csv(
        "synthetic_transactions_openai.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"✓ {category} fertig | "
        f"Gesamt gespeichert: {len(synthetic_rows)}"
    )

print("\n==============================")
print("GENERIERUNG ABGESCHLOSSEN")
print("==============================")
print("Synthetische Beispiele:", len(synthetic_rows))

synthetic_openai = pd.DataFrame(synthetic_rows)

display(synthetic_openai.head(20))

In [ ]:
# ============================================================
# QUALITÄTSPRÜFUNG DER SYNTHETISCHEN DATEN
# ============================================================

# dieselbe Bereinigung wie bei den realen Daten anwenden
synthetic_openai["clean_description"] = (
    synthetic_openai["clean_description"]
    .apply(clean_description)
)

print("Synthetische Beispiele insgesamt:", len(synthetic_openai))

print(
    "Leere Beschreibungen:",
    (synthetic_openai["clean_description"].str.len() == 0).sum()
)

print(
    "Duplikate innerhalb der synthetischen Daten:",
    synthetic_openai["clean_description"].duplicated().sum()
)

# Überschneidung mit Validation
val_overlap = set(
    synthetic_openai["clean_description"]
).intersection(
    set(val_grouped["clean_description"])
)

# Überschneidung mit Test
test_overlap = set(
    synthetic_openai["clean_description"]
).intersection(
    set(test_grouped["clean_description"])
)

print("Überschneidung mit Validation:", len(val_overlap))
print("Überschneidung mit Test:", len(test_overlap))

print(
    "Kategorien mit synthetischen Beispielen:",
    synthetic_openai["categoryName"].nunique()
)

In [ ]:
# ============================================================
# FINALEN TRAININGSDATENSATZ ERSTELLEN
# ============================================================

train_real = train_grouped[
    ["clean_description", "categoryName"]
].copy()

train_synthetic = synthetic_openai[
    ["clean_description", "categoryName"]
].copy()

train_final = pd.concat(
    [train_real, train_synthetic],
    ignore_index=True
)

# Labels mit demselben Encoder erzeugen
train_final["label"] = label_encoder_v2.transform(
    train_final["categoryName"]
)

print("Reale Trainingsbeispiele:", len(train_real))
print("Synthetische Trainingsbeispiele:", len(train_synthetic))
print("Finales Training:", len(train_final))
print("Validation:", len(val_grouped))
print("Test:", len(test_grouped))
print("Kategorien im Training:", train_final["categoryName"].nunique())

display(
    train_final["categoryName"]
    .value_counts()
    .sort_index()
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

finbert_model_name = "ProsusAI/finbert"

finbert_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)

finbert_model = AutoModelForSequenceClassification.from_pretrained(
    finbert_model_name,
    num_labels=37,
    ignore_mismatched_sizes=True
)

finbert_model.to(device)

print("FinBERT geladen.")

In [ ]:
train_encodings_fb = finbert_tokenizer(
    train_final["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings_fb = finbert_tokenizer(
    val_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings_fb = finbert_tokenizer(
    test_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("FinBERT-Tokenisierung abgeschlossen.")
print("Training:", len(train_encodings_fb["input_ids"]))
print("Validation:", len(val_encodings_fb["input_ids"]))
print("Test:", len(test_encodings_fb["input_ids"]))

In [ ]:
train_dataset_fb = TransactionDataset(
    train_encodings_fb,
    train_final["label"].tolist()
)

val_dataset_fb = TransactionDataset(
    val_encodings_fb,
    val_grouped["label"].tolist()
)

test_dataset_fb = TransactionDataset(
    test_encodings_fb,
    test_grouped["label"].tolist()
)

print("FinBERT-Datensätze erstellt.")
print("Training:", len(train_dataset_fb))
print("Validation:", len(val_dataset_fb))
print("Test:", len(test_dataset_fb))

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from copy import deepcopy
from tqdm.auto import tqdm

# ============================================================
# KLASSENGEWICHTE
# ============================================================

classes = np.arange(37)

class_weights_fb = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_final["label"].to_numpy()
)

class_weights_fb = torch.tensor(
    class_weights_fb,
    dtype=torch.float
).to(device)


# ============================================================
# FOCAL LOSS
# ============================================================

class FocalLossFinal(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        return focal_loss.mean()


criterion_fb = FocalLossFinal(
    alpha=class_weights_fb,
    gamma=2.0
)


# ============================================================
# DATALOADER
# ============================================================

train_loader_fb = DataLoader(
    train_dataset_fb,
    batch_size=16,
    shuffle=True
)

val_loader_fb = DataLoader(
    val_dataset_fb,
    batch_size=16,
    shuffle=False
)

test_loader_fb = DataLoader(
    test_dataset_fb,
    batch_size=16,
    shuffle=False
)


# ============================================================
# OPTIMIZER + SCHEDULER
# ============================================================

optimizer_fb = AdamW(
    finbert_model.parameters(),
    lr=2e-5
)

epochs_fb = 3

total_steps_fb = len(train_loader_fb) * epochs_fb
warmup_steps_fb = int(0.10 * total_steps_fb)

scheduler_fb = get_linear_schedule_with_warmup(
    optimizer_fb,
    num_warmup_steps=warmup_steps_fb,
    num_training_steps=total_steps_fb
)

print("Training vorbereitet.")
print("Batches pro Epoche:", len(train_loader_fb))
print("Training Steps gesamt:", total_steps_fb)
print("Warmup Steps:", warmup_steps_fb)

In [ ]:
best_val_accuracy_fb = 0.0
best_model_state_fb = None
best_epoch_fb = None

for epoch in range(epochs_fb):

    print(f"\n===== FinBERT Epoche {epoch + 1}/{epochs_fb} =====")

    # TRAINING
    finbert_model.train()
    total_train_loss = 0

    progress_bar = tqdm(
        train_loader_fb,
        desc="Training FinBERT"
    )

    for batch in progress_bar:

        optimizer_fb.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = finbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_fb(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer_fb.step()
        scheduler_fb.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = total_train_loss / len(train_loader_fb)

    # VALIDATION
    finbert_model.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_fb:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = finbert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_fb(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader_fb)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    # besten Checkpoint speichern
    if val_accuracy > best_val_accuracy_fb:

        best_val_accuracy_fb = val_accuracy
        best_epoch_fb = epoch + 1

        best_model_state_fb = deepcopy(
            finbert_model.state_dict()
        )

        print("-> Neuer bester Checkpoint gespeichert.")


# besten Checkpoint wieder laden
finbert_model.load_state_dict(best_model_state_fb)

print("\nBestes FinBERT-Modell geladen.")
print("Beste Epoche:", best_epoch_fb)
print(
    "Beste Validation Accuracy:",
    round(best_val_accuracy_fb, 4)
)

In [ ]:
class FocalLossCorrect(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)

        pt = probs.gather(
            1,
            targets.unsqueeze(1)
        ).squeeze(1)

        pt = pt.clamp(min=1e-8, max=1.0)

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
        else:
            alpha_t = 1.0

        loss = (
            -alpha_t
            * (1 - pt) ** self.gamma
            * torch.log(pt)
        )

        return loss.mean()

In [ ]:
criterion_fb = FocalLossCorrect(
    alpha=class_weights_fb,
    gamma=2.0
)

In [ ]:
from transformers import AutoModelForSequenceClassification

finbert_model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=37,
    ignore_mismatched_sizes=True
)

finbert_model.to(device)

print("Frisches FinBERT geladen.")

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer_fb = AdamW(
    finbert_model.parameters(),
    lr=2e-5
)

epochs_fb = 3

total_steps_fb = len(train_loader_fb) * epochs_fb
warmup_steps_fb = int(0.10 * total_steps_fb)

scheduler_fb = get_linear_schedule_with_warmup(
    optimizer_fb,
    num_warmup_steps=warmup_steps_fb,
    num_training_steps=total_steps_fb
)

print("Neues Training vorbereitet.")
print("Training Steps:", total_steps_fb)
print("Warmup Steps:", warmup_steps_fb)

In [ ]:
best_val_accuracy_fb = 0.0
best_model_state_fb = None
best_epoch_fb = None

for epoch in range(epochs_fb):

    print(f"\n===== FinBERT Epoche {epoch + 1}/{epochs_fb} =====")

    # TRAINING
    finbert_model.train()
    total_train_loss = 0

    progress_bar = tqdm(
        train_loader_fb,
        desc="Training FinBERT"
    )

    for batch in progress_bar:

        optimizer_fb.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = finbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_fb(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer_fb.step()
        scheduler_fb.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = total_train_loss / len(train_loader_fb)

    # VALIDATION
    finbert_model.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_fb:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = finbert_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_fb(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader_fb)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    if val_accuracy > best_val_accuracy_fb:

        best_val_accuracy_fb = val_accuracy
        best_epoch_fb = epoch + 1

        best_model_state_fb = deepcopy(
            finbert_model.state_dict()
        )

        print("-> Neuer bester Checkpoint gespeichert.")


finbert_model.load_state_dict(best_model_state_fb)

print("\nBestes FinBERT-Modell geladen.")
print("Beste Epoche:", best_epoch_fb)
print(
    "Beste Validation Accuracy:",
    round(best_val_accuracy_fb, 4)
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

german_model_name = "scherrmann/GermanFinBert_FP"

german_tokenizer = AutoTokenizer.from_pretrained(german_model_name)

german_model = AutoModelForSequenceClassification.from_pretrained(
    german_model_name,
    num_labels=37,
    ignore_mismatched_sizes=True
)

german_model.to(device)

print("GermanFinBERT geladen.")

In [ ]:
train_encodings_gfb = german_tokenizer(
    train_final["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings_gfb = german_tokenizer(
    val_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings_gfb = german_tokenizer(
    test_grouped["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("GermanFinBERT-Tokenisierung abgeschlossen.")
print("Training:", len(train_encodings_gfb["input_ids"]))
print("Validation:", len(val_encodings_gfb["input_ids"]))
print("Test:", len(test_encodings_gfb["input_ids"]))

In [ ]:
train_dataset_gfb = TransactionDataset(
    train_encodings_gfb,
    train_final["label"].tolist()
)

val_dataset_gfb = TransactionDataset(
    val_encodings_gfb,
    val_grouped["label"].tolist()
)

test_dataset_gfb = TransactionDataset(
    test_encodings_gfb,
    test_grouped["label"].tolist()
)

print("GermanFinBERT-Datensätze erstellt.")
print("Training:", len(train_dataset_gfb))
print("Validation:", len(val_dataset_gfb))
print("Test:", len(test_dataset_gfb))

In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# dieselben Klassengewichte wie beim FinBERT-Lauf
criterion_gfb = FocalLossCorrect(
    alpha=class_weights_fb,
    gamma=2.0
)

train_loader_gfb = DataLoader(
    train_dataset_gfb,
    batch_size=16,
    shuffle=True
)

val_loader_gfb = DataLoader(
    val_dataset_gfb,
    batch_size=16,
    shuffle=False
)

test_loader_gfb = DataLoader(
    test_dataset_gfb,
    batch_size=16,
    shuffle=False
)

optimizer_gfb = AdamW(
    german_model.parameters(),
    lr=2e-5
)

epochs_gfb = 3

total_steps_gfb = len(train_loader_gfb) * epochs_gfb
warmup_steps_gfb = int(0.10 * total_steps_gfb)

scheduler_gfb = get_linear_schedule_with_warmup(
    optimizer_gfb,
    num_warmup_steps=warmup_steps_gfb,
    num_training_steps=total_steps_gfb
)

print("GermanFinBERT-Training vorbereitet.")
print("Batches pro Epoche:", len(train_loader_gfb))
print("Training Steps gesamt:", total_steps_gfb)
print("Warmup Steps:", warmup_steps_gfb)

In [ ]:
best_val_accuracy_gfb = 0.0
best_model_state_gfb = None
best_epoch_gfb = None

for epoch in range(epochs_gfb):

    print(f"\n===== GermanFinBERT Epoche {epoch + 1}/{epochs_gfb} =====")

    # TRAINING
    german_model.train()
    total_train_loss = 0

    progress_bar = tqdm(
        train_loader_gfb,
        desc="Training GermanFinBERT"
    )

    for batch in progress_bar:

        optimizer_gfb.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = german_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion_gfb(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer_gfb.step()
        scheduler_gfb.step()

        total_train_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = total_train_loss / len(train_loader_gfb)

    # VALIDATION
    german_model.eval()

    correct = 0
    total = 0
    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader_gfb:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = german_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion_gfb(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    avg_val_loss = total_val_loss / len(val_loader_gfb)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    if val_accuracy > best_val_accuracy_gfb:

        best_val_accuracy_gfb = val_accuracy
        best_epoch_gfb = epoch + 1

        best_model_state_gfb = deepcopy(
            german_model.state_dict()
        )

        print("-> Neuer bester Checkpoint gespeichert.")


german_model.load_state_dict(best_model_state_gfb)

print("\nBestes GermanFinBERT-Modell geladen.")
print("Beste Epoche:", best_epoch_gfb)
print(
    "Beste Validation Accuracy:",
    round(best_val_accuracy_gfb, 4)
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import torch

german_model.eval()

test_predictions = []
test_true_labels = []
test_probabilities = []

with torch.no_grad():

    for batch in test_loader_gfb:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = german_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)

        predictions = torch.argmax(
            probs,
            dim=1
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_true_labels.extend(
            labels.cpu().numpy()
        )

        test_probabilities.extend(
            probs.cpu().numpy()
        )


test_predictions = np.array(test_predictions)
test_true_labels = np.array(test_true_labels)
test_probabilities = np.array(test_probabilities)


# Top-1 Accuracy
test_accuracy_gfb = accuracy_score(
    test_true_labels,
    test_predictions
)

# Macro-F1
test_macro_f1_gfb = f1_score(
    test_true_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

# Weighted-F1
test_weighted_f1_gfb = f1_score(
    test_true_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)

# Top-2 Accuracy
top2_predictions = np.argsort(
    test_probabilities,
    axis=1
)[:, -2:]

top2_correct = [
    true_label in top2
    for true_label, top2
    in zip(test_true_labels, top2_predictions)
]

test_top2_accuracy_gfb = np.mean(top2_correct)


print("==============================")
print("GERMANFINBERT – TESTERGEBNISSE")
print("==============================")
print(f"Test Accuracy:      {test_accuracy_gfb:.4f}")
print(f"Test Macro-F1:      {test_macro_f1_gfb:.4f}")
print(f"Test Weighted-F1:   {test_weighted_f1_gfb:.4f}")
print(f"Test Top-2 Accuracy:{test_top2_accuracy_gfb:.4f}")

In [ ]:
import torch
import torch.nn.functional as F

# ============================================================
# VALIDATION LOGITS SAMMELN
# ============================================================

german_model.eval()

val_logits_list = []
val_labels_list = []

with torch.no_grad():

    for batch in val_loader_gfb:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = german_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        val_logits_list.append(
            outputs.logits.detach()
        )

        val_labels_list.append(
            labels.detach()
        )


val_logits = torch.cat(val_logits_list)
val_labels = torch.cat(val_labels_list)


# ============================================================
# TEMPERATURE SCALING
# ============================================================

log_temperature = torch.nn.Parameter(
    torch.zeros(1, device=device)
)

optimizer_temp = torch.optim.LBFGS(
    [log_temperature],
    lr=0.01,
    max_iter=100
)

nll_before = F.cross_entropy(
    val_logits,
    val_labels
).item()


def closure():

    optimizer_temp.zero_grad()

    temperature = torch.exp(log_temperature)

    loss = F.cross_entropy(
        val_logits / temperature,
        val_labels
    )

    loss.backward()

    return loss


optimizer_temp.step(closure)

temperature_gfb = torch.exp(
    log_temperature
).item()

nll_after = F.cross_entropy(
    val_logits / temperature_gfb,
    val_labels
).item()


print("==============================")
print("TEMPERATURE SCALING")
print("==============================")
print(f"Temperature: {temperature_gfb:.4f}")
print(f"NLL vorher:  {nll_before:.4f}")
print(f"NLL nachher: {nll_after:.4f}")

In [ ]:
# ============================================================
# KALIBRIERTE CONFIDENCE-AUSWERTUNG AUF DEM TESTSET
# ============================================================

test_logits_list = []
test_labels_list = []

german_model.eval()

with torch.no_grad():

    for batch in test_loader_gfb:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = german_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        test_logits_list.append(
            outputs.logits.detach()
        )

        test_labels_list.append(
            labels.detach()
        )


test_logits = torch.cat(test_logits_list)
test_labels_tensor = torch.cat(test_labels_list)

# Temperature Scaling anwenden
calibrated_test_probs = torch.softmax(
    test_logits / temperature_gfb,
    dim=1
)

test_confidences, test_predicted = torch.max(
    calibrated_test_probs,
    dim=1
)

test_correct = (
    test_predicted == test_labels_tensor
)


# ============================================================
# CONFIDENCE-SCHWELLEN AUSWERTEN
# ============================================================

thresholds = [
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95
]

print("======================================================")
print("CONFIDENCE | FÄLLE | COVERAGE | ACCURACY")
print("======================================================")

for threshold in thresholds:

    mask = test_confidences >= threshold
    n_cases = mask.sum().item()

    if n_cases > 0:

        selective_accuracy = (
            test_correct[mask]
            .float()
            .mean()
            .item()
        )

        coverage = n_cases / len(test_labels_tensor)

        print(
            f">= {threshold:.2f}     | "
            f"{n_cases:4d}  | "
            f"{coverage:7.2%} | "
            f"{selective_accuracy:7.2%}"
        )

    else:

        print(
            f">= {threshold:.2f}     | "
            f"   0  | "
            f"  0.00% | "
            f"   -"
        )

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report

# Kategorienamen zurückübersetzen
true_names_test = label_encoder_v2.inverse_transform(
    test_labels_tensor.cpu().numpy()
)

pred_names_test = label_encoder_v2.inverse_transform(
    test_predicted.cpu().numpy()
)

test_analysis = pd.DataFrame({
    "true_category": true_names_test,
    "predicted_category": pred_names_test,
    "correct": true_names_test == pred_names_test,
    "confidence": test_confidences.cpu().numpy()
})

# Trefferquote pro Kategorie
category_performance = (
    test_analysis
    .groupby("true_category")
    .agg(
        cases=("correct", "size"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        avg_confidence=("confidence", "mean")
    )
    .sort_values(["accuracy", "cases"], ascending=[True, False])
)

category_performance["accuracy"] *= 100
category_performance["avg_confidence"] *= 100

print("==============================================")
print("TEST-LEISTUNG PRO KATEGORIE")
print("==============================================")

display(category_performance)

In [ ]:
test_structure = (
    test_grouped
    .groupby("categoryName")
    .agg(
        transactions=("clean_description", "size"),
        unique_texts=("clean_description", "nunique"),
        unique_groups=("group_id", "nunique")
    )
    .sort_values("transactions", ascending=False)
)

print("==============================================")
print("STRUKTUR DES TESTSETS")
print("==============================================")

display(test_structure)

print("\nGesamt:")
print("Transaktionen:", len(test_grouped))
print("Unterschiedliche Texte:", test_grouped["clean_description"].nunique())
print("Unterschiedliche Gruppen:", test_grouped["group_id"].nunique())

In [ ]:
# ============================================================
# GROUP-LEVEL-EVALUATION AUF DEM TESTSET
# ============================================================

test_group_eval = test_grouped[
    ["group_id", "clean_description", "categoryName"]
].reset_index(drop=True).copy()

test_group_eval["predicted_label"] = test_predicted.cpu().numpy()
test_group_eval["true_label"] = test_labels_tensor.cpu().numpy()
test_group_eval["confidence"] = test_confidences.cpu().numpy()

# Top-2 Vorhersagen speichern
top2_test = torch.topk(
    calibrated_test_probs,
    k=2,
    dim=1
).indices.cpu().numpy()

test_group_eval["top2_correct"] = [
    true_label in top2
    for true_label, top2
    in zip(
        test_group_eval["true_label"],
        top2_test
    )
]

# Da jede Gruppe nur einen eindeutigen Text enthält,
# reicht eine Zeile pro group_id
group_results = (
    test_group_eval
    .drop_duplicates(subset="group_id")
    .copy()
)

group_results["correct"] = (
    group_results["predicted_label"]
    == group_results["true_label"]
)

group_accuracy = group_results["correct"].mean()
group_top2_accuracy = group_results["top2_correct"].mean()

print("==============================================")
print("GROUP-LEVEL TESTERGEBNISSE")
print("==============================================")
print("Unabhängige Testmuster:", len(group_results))
print("Davon korrekt:", group_results["correct"].sum())
print(f"Group Top-1 Accuracy: {group_accuracy:.4f}")
print(f"Group Top-2 Accuracy: {group_top2_accuracy:.4f}")

In [ ]:
# ============================================================
# GROUP-LEVEL-EVALUATION AUF DER VALIDATION
# ============================================================

german_model.eval()

val_logits_group = []
val_labels_group = []

with torch.no_grad():

    for batch in val_loader_gfb:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = german_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        val_logits_group.append(outputs.logits.detach())
        val_labels_group.append(labels.detach())


val_logits_group = torch.cat(val_logits_group)
val_labels_group = torch.cat(val_labels_group)

# kalibrierte Wahrscheinlichkeiten
val_probs_group = torch.softmax(
    val_logits_group / temperature_gfb,
    dim=1
)

val_confidences_group, val_predicted_group = torch.max(
    val_probs_group,
    dim=1
)

top2_val = torch.topk(
    val_probs_group,
    k=2,
    dim=1
).indices.cpu().numpy()


val_group_eval = val_grouped[
    ["group_id", "clean_description", "categoryName"]
].reset_index(drop=True).copy()

val_group_eval["predicted_label"] = (
    val_predicted_group.cpu().numpy()
)

val_group_eval["true_label"] = (
    val_labels_group.cpu().numpy()
)

val_group_eval["confidence"] = (
    val_confidences_group.cpu().numpy()
)

val_group_eval["top2_correct"] = [
    true_label in top2
    for true_label, top2 in zip(
        val_group_eval["true_label"],
        top2_val
    )
]

val_group_results = (
    val_group_eval
    .drop_duplicates(subset="group_id")
    .copy()
)

val_group_results["correct"] = (
    val_group_results["predicted_label"]
    == val_group_results["true_label"]
)

val_group_accuracy = (
    val_group_results["correct"].mean()
)

val_group_top2_accuracy = (
    val_group_results["top2_correct"].mean()
)

print("==============================================")
print("GROUP-LEVEL VALIDATION")
print("==============================================")
print("Unabhängige Validierungsmuster:", len(val_group_results))
print("Davon korrekt:", val_group_results["correct"].sum())
print(f"Group Top-1 Accuracy: {val_group_accuracy:.4f}")
print(f"Group Top-2 Accuracy: {val_group_top2_accuracy:.4f}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

# ============================================================
# REALE DATEN FÜR 5-FOLD CROSS-VALIDATION VORBEREITEN
# ============================================================

cv_data = data[
    ["clean_description", "categoryName"]
].dropna().copy()

# Leere Beschreibungen entfernen
cv_data = cv_data[
    cv_data["clean_description"].str.strip() != ""
].reset_index(drop=True)

# Einheitlicher Label Encoder für alle 37 Kategorien
cv_label_encoder = LabelEncoder()

cv_data["label"] = cv_label_encoder.fit_transform(
    cv_data["categoryName"]
)

print("Reale Transaktionen:", len(cv_data))
print("Kategorien:", len(cv_label_encoder.classes_))
print("Kleinste Klasse:", cv_data["categoryName"].value_counts().min())

# ============================================================
# 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(
    skf.split(
        cv_data["clean_description"],
        cv_data["label"]
    )
)

print("\n5 Folds erstellt.")

for i, (train_idx, val_idx) in enumerate(folds, start=1):

    fold_train = cv_data.iloc[train_idx]
    fold_val = cv_data.iloc[val_idx]

    print(f"\nFold {i}")
    print("Training:", len(fold_train))
    print("Validation:", len(fold_val))
    print(
        "Kategorien Training:",
        fold_train["categoryName"].nunique()
    )
    print(
        "Kategorien Validation:",
        fold_val["categoryName"].nunique()
    )

In [ ]:
# ============================================================
# PAPER-NAHER AUGMENTATIONSPLAN – FOLD 1
# ============================================================

train_idx_fold1, val_idx_fold1 = folds[0]

fold1_train = cv_data.iloc[train_idx_fold1].copy()
fold1_val = cv_data.iloc[val_idx_fold1].copy()

fold1_counts = (
    fold1_train["categoryName"]
    .value_counts()
    .sort_values()
)

print("Klassenverteilung Fold 1:")
display(fold1_counts.to_frame("real_train"))

print("\nGrößte Klasse:", fold1_counts.max())
print("Kleinste Klasse:", fold1_counts.min())
print("Kategorien:", len(fold1_counts))

In [ ]:
# ============================================================
# PAPER-NAHER INVERSE-FREQUENCY AUGMENTATIONSPLAN – FOLD 1
# ============================================================

max_count_fold1 = fold1_counts.max()

augmentation_plan_fold1 = pd.DataFrame({
    "real_train": fold1_counts
})

augmentation_plan_fold1["synthetic_ratio"] = (
    max_count_fold1
    / augmentation_plan_fold1["real_train"]
).round()

# Paper: maximal ca. 30x für Minderheitsklassen
augmentation_plan_fold1["synthetic_ratio"] = (
    augmentation_plan_fold1["synthetic_ratio"]
    .clip(lower=1, upper=30)
    .astype(int)
)

augmentation_plan_fold1["synthetic_needed"] = (
    augmentation_plan_fold1["real_train"]
    * augmentation_plan_fold1["synthetic_ratio"]
)

augmentation_plan_fold1["final_size"] = (
    augmentation_plan_fold1["real_train"]
    + augmentation_plan_fold1["synthetic_needed"]
)

augmentation_plan_fold1 = augmentation_plan_fold1.sort_values(
    "real_train"
)

display(augmentation_plan_fold1)

print("\nGesamt reale Trainingsbeispiele:", len(fold1_train))
print(
    "Geplante synthetische Beispiele:",
    augmentation_plan_fold1["synthetic_needed"].sum()
)
print(
    "Training nach Augmentation:",
    len(fold1_train)
    + augmentation_plan_fold1["synthetic_needed"].sum()
)

In [ ]:
# ============================================================
# SYNTHETISCHE BEISPIELE AUF REALE AUSGANGSTEXTE VERTEILEN
# ============================================================

generation_plan_rows = []

rng = np.random.RandomState(42)

for category, row in augmentation_plan_fold1.iterrows():

    synthetic_needed = int(row["synthetic_needed"])

    real_examples = (
        fold1_train.loc[
            fold1_train["categoryName"] == category,
            "clean_description"
        ]
        .tolist()
    )

    # Reihenfolge reproduzierbar mischen
    rng.shuffle(real_examples)

    n_real = len(real_examples)

    # möglichst gleichmäßig auf reale Beispiele verteilen
    base_n = synthetic_needed // n_real
    remainder = synthetic_needed % n_real

    for i, text in enumerate(real_examples):

        n_generate = base_n

        # Rest auf die ersten Beispiele verteilen
        if i < remainder:
            n_generate += 1

        generation_plan_rows.append({
            "categoryName": category,
            "source_text": text,
            "n_generate": n_generate
        })


generation_plan_fold1 = pd.DataFrame(
    generation_plan_rows
)

print("Reale Ausgangstexte:", len(generation_plan_fold1))
print(
    "Geplante synthetische Beispiele:",
    generation_plan_fold1["n_generate"].sum()
)

display(
    generation_plan_fold1.head(30)
)

In [ ]:
# ============================================================
# GENERIERUNGSPLAN AUF EINDEUTIGE REALE MUSTER VERTEILEN
# ============================================================

generation_plan_rows = []

rng = np.random.RandomState(42)

for category, row in augmentation_plan_fold1.iterrows():

    synthetic_needed = int(row["synthetic_needed"])

    # Nur EINDEUTIGE reale Ausgangsbeschreibungen
    real_examples = (
        fold1_train.loc[
            fold1_train["categoryName"] == category,
            "clean_description"
        ]
        .drop_duplicates()
        .tolist()
    )

    rng.shuffle(real_examples)

    n_unique = len(real_examples)

    base_n = synthetic_needed // n_unique
    remainder = synthetic_needed % n_unique

    for i, text in enumerate(real_examples):

        n_generate = base_n

        if i < remainder:
            n_generate += 1

        generation_plan_rows.append({
            "categoryName": category,
            "source_text": text,
            "n_generate": n_generate
        })


generation_plan_fold1 = pd.DataFrame(
    generation_plan_rows
)

print("Eindeutige reale Ausgangsmuster:",
      len(generation_plan_fold1))

print(
    "Geplante synthetische Beispiele:",
    generation_plan_fold1["n_generate"].sum()
)

display(
    generation_plan_fold1.head(40)
)

In [ ]:
import json

def generate_paper_variants(source_text, category, n):

    prompt = f"""
Du erzeugst synthetische Banktransaktionsbeschreibungen
für das Training eines Klassifikationsmodells.

Reale Ausgangstransaktion:
"{source_text}"

Zugehörige Kategorie:
"{category}"

Erzeuge genau {n} neue Varianten dieser Transaktion.

WICHTIG:
- Die semantische Bedeutung der Ausgangstransaktion muss erhalten bleiben.
- Alle Varianten müssen eindeutig zur Kategorie "{category}" gehören.
- Erzeuge keine Transaktion, die fachlich zu einer anderen Kategorie gehört.
- Die Ausgangstransaktion darf nicht wortwörtlich kopiert werden.
- Verwende unterschiedliche realistische Formulierungen, Händler-, Zahlungs-
  oder Buchungsbezeichnungen, soweit sie zur Bedeutung passen.
- Formuliere kurz und banktypisch.
- Keine Erklärungen.
- Keine Nummerierung.
- Ausschließlich ein gültiges JSON-Array mit genau {n} Strings ausgeben.
"""

    response = client.chat.completions.create(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7
    )

    raw_text = response.choices[0].message.content.strip()

    generated = json.loads(raw_text)

    if not isinstance(generated, list):
        raise ValueError("Antwort ist keine Liste.")

    if len(generated) != n:
        raise ValueError(
            f"Erwartet: {n}, erhalten: {len(generated)}"
        )

    if not all(isinstance(x, str) for x in generated):
        raise ValueError("Nicht alle Elemente sind Strings.")

    return generated

In [ ]:
test_row = generation_plan_fold1.iloc[0]

print("Kategorie:", test_row["categoryName"])
print("Ausgangstext:", test_row["source_text"])

test_variants = generate_paper_variants(
    source_text=test_row["source_text"],
    category=test_row["categoryName"],
    n=10
)

for i, text in enumerate(test_variants, 1):
    print(f"{i}: {text}")

In [ ]:
import json

def generate_paper_variants(source_text, category, n):

    prompt = f"""
Du erzeugst synthetische Banktransaktionsbeschreibungen
für das Training eines Klassifikationsmodells.

Reale Ausgangstransaktion:
"{source_text}"

Zugehörige Kategorie:
"{category}"

Erzeuge genau {n} neue Varianten dieser Transaktion.

WICHTIG:
- Die semantische Bedeutung der Ausgangstransaktion muss erhalten bleiben.
- Alle Varianten müssen eindeutig zur Kategorie "{category}" gehören.
- Erzeuge keine Transaktion, die fachlich zu einer anderen Kategorie gehört.
- Die Ausgangstransaktion darf nicht wortwörtlich kopiert werden.
- Verwende unterschiedliche realistische Formulierungen, Händler-, Zahlungs-
  oder Buchungsbezeichnungen, soweit sie zur Bedeutung passen.
- Formuliere kurz und banktypisch.
- Keine Erklärungen.
- Keine Nummerierung.
- Ausschließlich ein gültiges JSON-Array mit genau {n} Strings ausgeben.
"""

    response = client.chat.completions.create(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    raw_text = response.choices[0].message.content.strip()

    generated = json.loads(raw_text)

    if not isinstance(generated, list):
        raise ValueError("Antwort ist keine Liste.")

    if len(generated) != n:
        raise ValueError(
            f"Erwartet: {n}, erhalten: {len(generated)}"
        )

    if not all(isinstance(x, str) for x in generated):
        raise ValueError("Nicht alle Elemente sind Strings.")

    return generated


# ============================================================
# TEST MIT EINEM REALEN AUSGANGSMUSTER
# ============================================================

test_row = generation_plan_fold1.iloc[0]

print("Kategorie:", test_row["categoryName"])
print("Ausgangstext:", test_row["source_text"])

test_variants = generate_paper_variants(
    source_text=test_row["source_text"],
    category=test_row["categoryName"],
    n=10
)

for i, text in enumerate(test_variants, 1):
    print(f"{i}: {text}")

In [ ]:
import pandas as pd
import time

synthetic_rows_fold1 = []

for idx, row in generation_plan_fold1.iterrows():

    category = row["categoryName"]
    source_text = row["source_text"]
    n_generate = int(row["n_generate"])

    print(
        f"[{idx + 1}/{len(generation_plan_fold1)}] "
        f"{category} | {source_text} | {n_generate} Varianten"
    )

    generated = generate_paper_variants(
        source_text=source_text,
        category=category,
        n=n_generate
    )

    for text in generated:
        synthetic_rows_fold1.append({
            "clean_description": text,
            "categoryName": category,
            "source_text": source_text,
            "synthetic": True
        })

    # Nach jedem Ausgangsmuster speichern
    synthetic_fold1 = pd.DataFrame(
        synthetic_rows_fold1
    )

    synthetic_fold1.to_csv(
        "synthetic_fold1_paper_style.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Gesamt gespeichert:",
        len(synthetic_rows_fold1)
    )

    time.sleep(0.2)


print("\n====================================")
print("GENERIERUNG FOLD 1 ABGESCHLOSSEN")
print("====================================")
print(
    "Synthetische Beispiele:",
    len(synthetic_rows_fold1)
)

synthetic_fold1 = pd.DataFrame(
    synthetic_rows_fold1
)

display(
    synthetic_fold1.head(20)
)

In [ ]:
import json

def generate_paper_variants(source_text, category, n):

    all_generated = []

    while len(all_generated) < n:

        missing = n - len(all_generated)

        prompt = f"""
Du erzeugst synthetische Banktransaktionsbeschreibungen
für das Training eines Klassifikationsmodells.

Reale Ausgangstransaktion:
"{source_text}"

Zugehörige Kategorie:
"{category}"

Erzeuge genau {missing} neue Varianten dieser Transaktion.

WICHTIG:
- Die semantische Bedeutung der Ausgangstransaktion muss erhalten bleiben.
- Alle Varianten müssen eindeutig zur Kategorie "{category}" gehören.
- Erzeuge keine Transaktion, die fachlich zu einer anderen Kategorie gehört.
- Die Ausgangstransaktion darf nicht wortwörtlich kopiert werden.
- Verwende unterschiedliche realistische Formulierungen, Händler-, Zahlungs-
  oder Buchungsbezeichnungen, soweit sie zur Bedeutung passen.
- Formuliere kurz und banktypisch.
- Keine Erklärungen.
- Keine Nummerierung.
- Ausschließlich ein gültiges JSON-Array mit Strings ausgeben.
"""

        response = client.chat.completions.create(
            model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        raw_text = response.choices[0].message.content.strip()
        generated = json.loads(raw_text)

        if not isinstance(generated, list):
            raise ValueError("Antwort ist keine Liste.")

        generated = [
            x.strip()
            for x in generated
            if isinstance(x, str) and x.strip()
        ]

        all_generated.extend(generated)

    # Falls etwas zu viel generiert wurde
    all_generated = all_generated[:n]

    return all_generated

In [ ]:
import pandas as pd
import time

synthetic_rows_fold1 = []

for pos, (_, row) in enumerate(
    generation_plan_fold1.iterrows(),
    start=1
):

    category = row["categoryName"]
    source_text = row["source_text"]
    n_generate = int(row["n_generate"])

    print(
        f"[{pos}/{len(generation_plan_fold1)}] "
        f"{category} | {source_text} | "
        f"{n_generate} Varianten"
    )

    generated = generate_paper_variants(
        source_text=source_text,
        category=category,
        n=n_generate
    )

    for text in generated:
        synthetic_rows_fold1.append({
            "clean_description": text,
            "categoryName": category,
            "source_text": source_text,
            "synthetic": True
        })

    synthetic_fold1 = pd.DataFrame(
        synthetic_rows_fold1
    )

    synthetic_fold1.to_csv(
        "synthetic_fold1_paper_style.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Gesamt gespeichert:",
        len(synthetic_rows_fold1)
    )

    time.sleep(0.2)


print("\n====================================")
print("GENERIERUNG FOLD 1 ABGESCHLOSSEN")
print("====================================")
print(
    "Synthetische Beispiele:",
    len(synthetic_rows_fold1)
)

synthetic_fold1 = pd.DataFrame(
    synthetic_rows_fold1
)

display(
    synthetic_fold1.head(20)
)

In [ ]:
# ============================================================
# QUALITÄTSKONTROLLE DER SYNTHETISCHEN DATEN – FOLD 1
# ============================================================

synthetic_fold1 = pd.read_csv(
    "synthetic_fold1_paper_style.csv"
)

# gleiche Cleaning-Funktion anwenden
synthetic_fold1["clean_description"] = (
    synthetic_fold1["clean_description"]
    .astype(str)
    .apply(clean_description)
)

# ------------------------------------------------------------
# 1. Grundlegende Checks
# ------------------------------------------------------------

total_synthetic = len(synthetic_fold1)

empty_count = (
    synthetic_fold1["clean_description"]
    .str.strip()
    .eq("")
    .sum()
)

duplicate_count = synthetic_fold1.duplicated(
    subset=["clean_description", "categoryName"]
).sum()

# ------------------------------------------------------------
# 2. Überschneidung mit Validation
# ------------------------------------------------------------

validation_pairs = set(
    zip(
        fold1_val["clean_description"],
        fold1_val["categoryName"]
    )
)

synthetic_pairs = set(
    zip(
        synthetic_fold1["clean_description"],
        synthetic_fold1["categoryName"]
    )
)

overlap_validation = len(
    synthetic_pairs & validation_pairs
)

# ------------------------------------------------------------
# 3. Überschneidung mit realem Training
# ------------------------------------------------------------

train_pairs = set(
    zip(
        fold1_train["clean_description"],
        fold1_train["categoryName"]
    )
)

overlap_training = len(
    synthetic_pairs & train_pairs
)

# ------------------------------------------------------------
# 4. Kategorieabdeckung
# ------------------------------------------------------------

synthetic_categories = (
    synthetic_fold1["categoryName"]
    .nunique()
)

# ------------------------------------------------------------
# Ausgabe
# ------------------------------------------------------------

print("==============================================")
print("QUALITÄTSCHECK SYNTHETISCHE DATEN – FOLD 1")
print("==============================================")

print("Synthetische Beispiele:", total_synthetic)
print("Leere Beschreibungen:", empty_count)
print("Doppelte Beispiele:", duplicate_count)
print("Overlap mit Validation:", overlap_validation)
print("Overlap mit realem Training:", overlap_training)
print("Kategorien:", synthetic_categories)

print("\nSynthetische Beispiele pro Kategorie:")

display(
    synthetic_fold1["categoryName"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# SYNTHETISCHE DATEN BEREINIGEN – FOLD 1
# ============================================================

synthetic_fold1_clean = synthetic_fold1.copy()

# 1. Leere Texte entfernen
synthetic_fold1_clean = synthetic_fold1_clean[
    synthetic_fold1_clean["clean_description"]
    .str.strip()
    .ne("")
].copy()

# 2. Duplikate innerhalb derselben Kategorie entfernen
synthetic_fold1_clean = synthetic_fold1_clean.drop_duplicates(
    subset=["clean_description", "categoryName"]
).copy()

# 3. Overlap mit Validation entfernen
synthetic_fold1_clean = synthetic_fold1_clean[
    ~synthetic_fold1_clean.apply(
        lambda row: (
            row["clean_description"],
            row["categoryName"]
        ) in validation_pairs,
        axis=1
    )
].copy()

# 4. Overlap mit realem Training entfernen
synthetic_fold1_clean = synthetic_fold1_clean[
    ~synthetic_fold1_clean.apply(
        lambda row: (
            row["clean_description"],
            row["categoryName"]
        ) in train_pairs,
        axis=1
    )
].copy()

synthetic_fold1_clean = synthetic_fold1_clean.reset_index(drop=True)

print("==============================================")
print("BEREINIGTE SYNTHETISCHE DATEN – FOLD 1")
print("==============================================")
print("Vorher:", len(synthetic_fold1))
print("Nachher:", len(synthetic_fold1_clean))
print("Entfernt:", len(synthetic_fold1) - len(synthetic_fold1_clean))
print("Kategorien:", synthetic_fold1_clean["categoryName"].nunique())


# ============================================================
# FINALES TRAININGSSET FÜR FOLD 1
# ============================================================

fold1_train_final = pd.concat(
    [
        fold1_train[
            ["clean_description", "categoryName", "label"]
        ].copy(),
        synthetic_fold1_clean[
            ["clean_description", "categoryName"]
        ].assign(
            label=lambda df: cv_label_encoder.transform(
                df["categoryName"]
            )
        )
    ],
    ignore_index=True
)

print("\n==============================================")
print("FINALES FOLD-1-TRAINING")
print("==============================================")
print("Reale Trainingsbeispiele:", len(fold1_train))
print("Synthetische Trainingsbeispiele:", len(synthetic_fold1_clean))
print("Gesamt:", len(fold1_train_final))
print("Kategorien:", fold1_train_final["categoryName"].nunique())

In [ ]:
# ============================================================
# PAPER-NAHER FINBERT-DATENSATZ – FOLD 1
# Nur synthetische Daten für das Fine-Tuning
# ============================================================

fold1_synthetic_train = synthetic_fold1_clean[
    ["clean_description", "categoryName"]
].copy()

fold1_synthetic_train["label"] = cv_label_encoder.transform(
    fold1_synthetic_train["categoryName"]
)

print("==============================================")
print("PAPER-NAHER FINE-TUNING-DATENSATZ – FOLD 1")
print("==============================================")
print("Synthetische Trainingsbeispiele:", len(fold1_synthetic_train))
print("Kategorien:", fold1_synthetic_train["categoryName"].nunique())
print("Reale Evaluation später:", len(fold1_val))

display(
    fold1_synthetic_train["categoryName"]
    .value_counts()
    .sort_index()
)

In [ ]:
from sklearn.model_selection import train_test_split

# ============================================================
# INTERNER STRATIFIZIERTER SPLIT DER SYNTHETISCHEN DATEN
# ============================================================

synthetic_train_fold1, synthetic_val_fold1 = train_test_split(
    fold1_synthetic_train,
    test_size=0.20,
    random_state=42,
    stratify=fold1_synthetic_train["label"]
)

synthetic_train_fold1 = synthetic_train_fold1.reset_index(drop=True)
synthetic_val_fold1 = synthetic_val_fold1.reset_index(drop=True)

print("==============================================")
print("SYNTHETISCHER TRAIN/VAL-SPLIT – FOLD 1")
print("==============================================")
print("Training:", len(synthetic_train_fold1))
print("Validation:", len(synthetic_val_fold1))
print("Kategorien Training:", synthetic_train_fold1["categoryName"].nunique())
print("Kategorien Validation:", synthetic_val_fold1["categoryName"].nunique())

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# FINBERT FÜR PAPER-NAHEN FOLD-1-LAUF
# ============================================================

paper_model_name = "ProsusAI/finbert"

paper_tokenizer = AutoTokenizer.from_pretrained(
    paper_model_name
)

paper_finbert = AutoModelForSequenceClassification.from_pretrained(
    paper_model_name,
    num_labels=37,
    ignore_mismatched_sizes=True
)

paper_finbert.to(device)

print("Frisches FinBERT für Fold 1 geladen.")


# ============================================================
# TOKENISIERUNG
# ============================================================

paper_train_encodings = paper_tokenizer(
    synthetic_train_fold1["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

paper_val_encodings = paper_tokenizer(
    synthetic_val_fold1["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

paper_real_eval_encodings = paper_tokenizer(
    fold1_val["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

print("\nTokenisierung abgeschlossen.")
print("Synthetisches Training:", len(paper_train_encodings["input_ids"]))
print("Synthetische Validation:", len(paper_val_encodings["input_ids"]))
print("Reale Fold-1-Evaluation:", len(paper_real_eval_encodings["input_ids"]))

In [ ]:
# ============================================================
# PYTORCH-DATENSÄTZE – PAPER-NAHER FOLD-1-LAUF
# ============================================================

paper_train_dataset = TransactionDataset(
    paper_train_encodings,
    synthetic_train_fold1["label"].tolist()
)

paper_val_dataset = TransactionDataset(
    paper_val_encodings,
    synthetic_val_fold1["label"].tolist()
)

paper_real_eval_dataset = TransactionDataset(
    paper_real_eval_encodings,
    fold1_val["label"].tolist()
)

print("PyTorch-Datensätze erstellt.")
print("Synthetisches Training:", len(paper_train_dataset))
print("Synthetische Validation:", len(paper_val_dataset))
print("Reale Fold-1-Evaluation:", len(paper_real_eval_dataset))

In [ ]:
# ============================================================
# WEIGHTED FOCAL LOSS + DATALOADER + OPTIMIZER + SCHEDULER
# PAPER-NAHER FOLD-1-LAUF
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup


# ------------------------------------------------------------
# 1. Klassen-Gewichte aus dem synthetischen Trainingsset
#    inverse Häufigkeit / balanced weighting
# ------------------------------------------------------------

num_labels = 37

train_labels_np = synthetic_train_fold1["label"].to_numpy()

class_counts = np.bincount(
    train_labels_np,
    minlength=num_labels
)

class_weights = (
    len(train_labels_np) /
    (num_labels * class_counts)
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("Klassen-Gewichte erstellt.")
print("Min. Gewicht:", class_weights_tensor.min().item())
print("Max. Gewicht:", class_weights_tensor.max().item())


# ------------------------------------------------------------
# 2. Weighted Focal Loss
#    L = -alpha_t * (1 - p_t)^gamma * log(p_t)
#    gamma = 2
# ------------------------------------------------------------

class WeightedFocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        probs = F.softmax(logits, dim=1)

        pt = probs.gather(
            1,
            targets.unsqueeze(1)
        ).squeeze(1)

        pt = pt.clamp(
            min=1e-8,
            max=1.0
        )

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
        else:
            alpha_t = 1.0

        loss = (
            -alpha_t
            * (1 - pt) ** self.gamma
            * torch.log(pt)
        )

        return loss.mean()


paper_focal_loss = WeightedFocalLoss(
    alpha=class_weights_tensor,
    gamma=2.0
)


# ------------------------------------------------------------
# 3. DataLoader
#    Paper: Batch Size = 16
# ------------------------------------------------------------

paper_train_loader = DataLoader(
    paper_train_dataset,
    batch_size=16,
    shuffle=True
)

paper_val_loader = DataLoader(
    paper_val_dataset,
    batch_size=16,
    shuffle=False
)

paper_real_eval_loader = DataLoader(
    paper_real_eval_dataset,
    batch_size=16,
    shuffle=False
)


# ------------------------------------------------------------
# 4. Optimizer
#    Paper: Learning Rate = 2e-5
# ------------------------------------------------------------

paper_optimizer = torch.optim.AdamW(
    paper_finbert.parameters(),
    lr=2e-5
)


# ------------------------------------------------------------
# 5. Scheduler
#    Paper:
#    Epochs = 3
#    Warmup Steps = 500
# ------------------------------------------------------------

paper_epochs = 3

paper_total_steps = (
    len(paper_train_loader)
    * paper_epochs
)

paper_warmup_steps = 500

paper_scheduler = get_linear_schedule_with_warmup(
    paper_optimizer,
    num_warmup_steps=paper_warmup_steps,
    num_training_steps=paper_total_steps
)


# ------------------------------------------------------------
# Kontrolle
# ------------------------------------------------------------

print("\n==============================================")
print("PAPER-TRAININGSKONFIGURATION – FOLD 1")
print("==============================================")

print("Trainingsbeispiele:", len(paper_train_dataset))
print("Validation:", len(paper_val_dataset))
print("Batch Size: 16")
print("Learning Rate: 2e-5")
print("Epochs:", paper_epochs)
print("Batches pro Epoche:", len(paper_train_loader))
print("Gesamte Trainingsschritte:", paper_total_steps)
print("Warmup Steps:", paper_warmup_steps)
print("Focal Loss Gamma: 2.0")

In [ ]:
# ============================================================
# FINBERT TRAINING – PAPER-NAHER FOLD-1-LAUF
# 3 Epochen, Weighted Focal Loss
# ============================================================

from tqdm.auto import tqdm
import copy

best_val_accuracy = 0.0
best_model_state = None

training_history = []


for epoch in range(paper_epochs):

    # ========================================================
    # TRAINING
    # ========================================================

    paper_finbert.train()

    total_train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(
        paper_train_loader,
        desc=f"Epoch {epoch + 1}/{paper_epochs} - Training"
    )

    for batch in train_progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        paper_optimizer.zero_grad()

        outputs = paper_finbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        loss = paper_focal_loss(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            paper_finbert.parameters(),
            max_norm=1.0
        )

        paper_optimizer.step()
        paper_scheduler.step()

        total_train_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

        train_progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    train_loss = (
        total_train_loss /
        len(paper_train_loader)
    )

    train_accuracy = (
        train_correct /
        train_total
    )


    # ========================================================
    # SYNTHETISCHE VALIDATION
    # ========================================================

    paper_finbert.eval()

    total_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch in paper_val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = paper_finbert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            loss = paper_focal_loss(
                logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)


    val_loss = (
        total_val_loss /
        len(paper_val_loader)
    )

    val_accuracy = (
        val_correct /
        val_total
    )


    # ========================================================
    # BESTES MODELL SPEICHERN
    # ========================================================

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        best_model_state = copy.deepcopy(
            paper_finbert.state_dict()
        )


    training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })


    print("\n==============================================")
    print(f"EPOCH {epoch + 1}/{paper_epochs}")
    print("==============================================")
    print(f"Train Loss:     {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss:       {val_loss:.4f}")
    print(f"Val Accuracy:   {val_accuracy:.4f}")
    print("==============================================\n")


# ============================================================
# BESTES MODELL WIEDERHERSTELLEN
# ============================================================

paper_finbert.load_state_dict(
    best_model_state
)

print("TRAINING ABGESCHLOSSEN")
print(
    f"Beste synthetische Validation Accuracy: "
    f"{best_val_accuracy:.4f}"
)

In [ ]:
# ============================================================
# REALE FOLD-1-EVALUATION – UNKALIBRIERT
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

paper_finbert.eval()

all_real_logits = []
all_real_labels = []

with torch.no_grad():

    for batch in paper_real_eval_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = paper_finbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        all_real_logits.append(
            outputs.logits.cpu()
        )

        all_real_labels.append(
            labels.cpu()
        )


# ------------------------------------------------------------
# Zusammenführen
# ------------------------------------------------------------

real_logits = torch.cat(
    all_real_logits,
    dim=0
)

real_labels = torch.cat(
    all_real_labels,
    dim=0
).numpy()


# ------------------------------------------------------------
# Wahrscheinlichkeiten
# ------------------------------------------------------------

real_probs = torch.softmax(
    real_logits,
    dim=1
).numpy()

real_predictions = np.argmax(
    real_probs,
    axis=1
)

real_confidence = np.max(
    real_probs,
    axis=1
)


# ------------------------------------------------------------
# Standard Accuracy
# ------------------------------------------------------------

real_accuracy = accuracy_score(
    real_labels,
    real_predictions
)

real_macro_f1 = f1_score(
    real_labels,
    real_predictions,
    average="macro",
    zero_division=0
)

real_weighted_f1 = f1_score(
    real_labels,
    real_predictions,
    average="weighted",
    zero_division=0
)


# ------------------------------------------------------------
# Top-2 Accuracy
# ------------------------------------------------------------

top2_predictions = np.argsort(
    real_probs,
    axis=1
)[:, -2:]

top2_correct = [
    true_label in top2
    for true_label, top2
    in zip(real_labels, top2_predictions)
]

real_top2_accuracy = np.mean(
    top2_correct
)


# ------------------------------------------------------------
# High-Confidence Accuracy > 0.8
# ------------------------------------------------------------

high_conf_mask = (
    real_confidence > 0.8
)

high_conf_count = (
    high_conf_mask.sum()
)

high_conf_coverage = (
    high_conf_mask.mean()
)

if high_conf_count > 0:

    high_conf_accuracy = accuracy_score(
        real_labels[high_conf_mask],
        real_predictions[high_conf_mask]
    )

else:

    high_conf_accuracy = np.nan


# ------------------------------------------------------------
# Top 50 % Confidence Accuracy
# ------------------------------------------------------------

top50_count = int(
    len(real_confidence) * 0.50
)

top50_indices = np.argsort(
    real_confidence
)[-top50_count:]

top50_accuracy = accuracy_score(
    real_labels[top50_indices],
    real_predictions[top50_indices]
)


# ------------------------------------------------------------
# AUSGABE
# ------------------------------------------------------------

print("==============================================")
print("REALE FOLD-1-EVALUATION – UNKALIBRIERT")
print("==============================================")

print(f"Reale Beispiele:              {len(real_labels)}")
print(f"Standard Accuracy:            {real_accuracy:.4f}")
print(f"Macro F1:                     {real_macro_f1:.4f}")
print(f"Weighted F1:                  {real_weighted_f1:.4f}")
print(f"Top-2 Accuracy:               {real_top2_accuracy:.4f}")

print("\n--- Confidence ---")

print(f"> 0.8 Fälle:                   {high_conf_count}")
print(f"> 0.8 Coverage:                {high_conf_coverage:.4f}")
print(f"> 0.8 Accuracy:                {high_conf_accuracy:.4f}")
print(f"Top-50%-Confidence Accuracy:  {top50_accuracy:.4f}")

In [ ]:
# ============================================================
# REALE KALIBRIERUNGSDATEN AUS FOLD-1-TRAINING
# ============================================================

from sklearn.model_selection import train_test_split

# 20 % der realen Fold-1-Trainingsdaten als Kalibrierungsset
_, fold1_calibration = train_test_split(
    fold1_train,
    test_size=0.20,
    random_state=42,
    stratify=fold1_train["label"]
)

fold1_calibration = fold1_calibration.reset_index(drop=True)

print("==============================================")
print("REALES KALIBRIERUNGSSET – FOLD 1")
print("==============================================")
print("Kalibrierungsbeispiele:", len(fold1_calibration))
print("Kategorien:", fold1_calibration["categoryName"].nunique())
print("Finale reale Evaluation bleibt:", len(fold1_val))

In [ ]:
# ============================================================
# KALIBRIERUNGSDATEN TOKENISIEREN + LOGITS BERECHNEN
# ============================================================

# Tokenisierung
paper_calibration_encodings = paper_tokenizer(
    fold1_calibration["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

# Dataset
paper_calibration_dataset = TransactionDataset(
    paper_calibration_encodings,
    fold1_calibration["label"].tolist()
)

# DataLoader
paper_calibration_loader = DataLoader(
    paper_calibration_dataset,
    batch_size=16,
    shuffle=False
)

# Logits berechnen
paper_finbert.eval()

calibration_logits_list = []
calibration_labels_list = []

with torch.no_grad():

    for batch in paper_calibration_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = paper_finbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        calibration_logits_list.append(
            outputs.logits.cpu()
        )

        calibration_labels_list.append(
            labels.cpu()
        )

calibration_logits = torch.cat(
    calibration_logits_list,
    dim=0
)

calibration_labels = torch.cat(
    calibration_labels_list,
    dim=0
)

print("==============================================")
print("KALIBRIERUNGSLOGITS – FOLD 1")
print("==============================================")
print("Logits Shape:", calibration_logits.shape)
print("Labels Shape:", calibration_labels.shape)
print("Kategorien:", len(torch.unique(calibration_labels)))

In [ ]:
# ============================================================
# PAPER-KALIBRIERUNG: TEMPERATUR T + BIAS b
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# 1. Kalibrierungsmodul
# ------------------------------------------------------------

class TemperatureBiasCalibrator(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        # log(T), damit T durch exp() immer > 0 bleibt
        self.log_temperature = nn.Parameter(
            torch.zeros(1)
        )

        # ein Bias-Wert pro Klasse
        self.bias = nn.Parameter(
            torch.zeros(num_classes)
        )

    def forward(self, logits):

        temperature = torch.exp(
            self.log_temperature
        )

        calibrated_logits = (
            logits / temperature
            + self.bias
        )

        return calibrated_logits

    def get_temperature(self):

        return torch.exp(
            self.log_temperature
        )


# ------------------------------------------------------------
# 2. Initialisieren
# ------------------------------------------------------------

paper_calibrator = TemperatureBiasCalibrator(
    num_classes=37
).to(device)

cal_logits_device = calibration_logits.to(device)
cal_labels_device = calibration_labels.to(device)


# ------------------------------------------------------------
# 3. NLL VOR Kalibrierung
# ------------------------------------------------------------

with torch.no_grad():

    nll_before = F.cross_entropy(
        cal_logits_device,
        cal_labels_device
    ).item()


# ------------------------------------------------------------
# 4. T und b optimieren
# ------------------------------------------------------------

optimizer_calibration = torch.optim.LBFGS(
    paper_calibrator.parameters(),
    lr=0.1,
    max_iter=500,
    line_search_fn="strong_wolfe"
)


def calibration_closure():

    optimizer_calibration.zero_grad()

    calibrated_logits = paper_calibrator(
        cal_logits_device
    )

    loss = F.cross_entropy(
        calibrated_logits,
        cal_labels_device
    )

    loss.backward()

    return loss


optimizer_calibration.step(
    calibration_closure
)


# ------------------------------------------------------------
# 5. NLL NACH Kalibrierung
# ------------------------------------------------------------

paper_calibrator.eval()

with torch.no_grad():

    calibrated_calibration_logits = paper_calibrator(
        cal_logits_device
    )

    nll_after = F.cross_entropy(
        calibrated_calibration_logits,
        cal_labels_device
    ).item()

    learned_temperature = (
        paper_calibrator
        .get_temperature()
        .item()
    )

    learned_bias = (
        paper_calibrator
        .bias
        .detach()
        .cpu()
        .numpy()
    )


# ------------------------------------------------------------
# AUSGABE
# ------------------------------------------------------------

print("==============================================")
print("PAPER-KALIBRIERUNG – FOLD 1")
print("==============================================")

print(f"NLL vorher:          {nll_before:.4f}")
print(f"NLL nachher:         {nll_after:.4f}")
print(f"Gelernte Temperatur: {learned_temperature:.4f}")

print("\nBias:")
print(learned_bias)

In [ ]:
# ============================================================
# REALE FOLD-1-EVALUATION – NACH KALIBRIERUNG
# ============================================================

import numpy as np
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    f1_score
)


# ------------------------------------------------------------
# ECE-Funktion
# ------------------------------------------------------------

def expected_calibration_error(
    probabilities,
    labels,
    n_bins=10
):

    confidences = np.max(
        probabilities,
        axis=1
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    accuracies = (
        predictions == labels
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == 0:
            mask = (
                (confidences >= lower)
                & (confidences <= upper)
            )
        else:
            mask = (
                (confidences > lower)
                & (confidences <= upper)
            )

        if np.any(mask):

            bin_accuracy = np.mean(
                accuracies[mask]
            )

            bin_confidence = np.mean(
                confidences[mask]
            )

            bin_weight = np.mean(
                mask
            )

            ece += (
                bin_weight
                * abs(
                    bin_accuracy
                    - bin_confidence
                )
            )

    return ece


# ------------------------------------------------------------
# 1. UNKALIBRIERTE ECE + NLL
# ------------------------------------------------------------

uncalibrated_nll = F.cross_entropy(
    real_logits,
    torch.tensor(
        real_labels,
        dtype=torch.long
    )
).item()

uncalibrated_ece = expected_calibration_error(
    real_probs,
    real_labels,
    n_bins=10
)


# ------------------------------------------------------------
# 2. GELERNTE KALIBRIERUNG AUF TEST-LOGITS ANWENDEN
# ------------------------------------------------------------

paper_calibrator.eval()

with torch.no_grad():

    calibrated_real_logits = paper_calibrator(
        real_logits.to(device)
    ).cpu()

calibrated_real_probs = torch.softmax(
    calibrated_real_logits,
    dim=1
).numpy()

calibrated_predictions = np.argmax(
    calibrated_real_probs,
    axis=1
)

calibrated_confidence = np.max(
    calibrated_real_probs,
    axis=1
)


# ------------------------------------------------------------
# 3. STANDARD-METRIKEN
# ------------------------------------------------------------

calibrated_accuracy = accuracy_score(
    real_labels,
    calibrated_predictions
)

calibrated_macro_f1 = f1_score(
    real_labels,
    calibrated_predictions,
    average="macro",
    zero_division=0
)

calibrated_weighted_f1 = f1_score(
    real_labels,
    calibrated_predictions,
    average="weighted",
    zero_division=0
)


# ------------------------------------------------------------
# 4. TOP-2 ACCURACY
# ------------------------------------------------------------

calibrated_top2 = np.argsort(
    calibrated_real_probs,
    axis=1
)[:, -2:]

calibrated_top2_accuracy = np.mean(
    [
        true_label in top2
        for true_label, top2
        in zip(
            real_labels,
            calibrated_top2
        )
    ]
)


# ------------------------------------------------------------
# 5. HIGH-CONFIDENCE > 0.8
# ------------------------------------------------------------

cal_high_mask = (
    calibrated_confidence > 0.8
)

cal_high_count = (
    cal_high_mask.sum()
)

cal_high_coverage = (
    cal_high_mask.mean()
)

if cal_high_count > 0:

    cal_high_accuracy = accuracy_score(
        real_labels[cal_high_mask],
        calibrated_predictions[cal_high_mask]
    )

else:

    cal_high_accuracy = np.nan


# ------------------------------------------------------------
# 6. TOP 50 % CONFIDENCE
# ------------------------------------------------------------

cal_top50_count = int(
    len(calibrated_confidence) * 0.50
)

cal_top50_indices = np.argsort(
    calibrated_confidence
)[-cal_top50_count:]

cal_top50_accuracy = accuracy_score(
    real_labels[cal_top50_indices],
    calibrated_predictions[cal_top50_indices]
)


# ------------------------------------------------------------
# 7. NLL + ECE NACH KALIBRIERUNG
# ------------------------------------------------------------

calibrated_nll = F.cross_entropy(
    calibrated_real_logits,
    torch.tensor(
        real_labels,
        dtype=torch.long
    )
).item()

calibrated_ece = expected_calibration_error(
    calibrated_real_probs,
    real_labels,
    n_bins=10
)


# ------------------------------------------------------------
# AUSGABE
# ------------------------------------------------------------

print("==============================================")
print("FOLD 1 – VORHER vs. NACHHER")
print("==============================================")

print("\nSTANDARD ACCURACY")
print(f"Vorher:  {real_accuracy:.4f}")
print(f"Nachher: {calibrated_accuracy:.4f}")

print("\nMACRO F1")
print(f"Vorher:  {real_macro_f1:.4f}")
print(f"Nachher: {calibrated_macro_f1:.4f}")

print("\nWEIGHTED F1")
print(f"Vorher:  {real_weighted_f1:.4f}")
print(f"Nachher: {calibrated_weighted_f1:.4f}")

print("\nTOP-2 ACCURACY")
print(f"Vorher:  {real_top2_accuracy:.4f}")
print(f"Nachher: {calibrated_top2_accuracy:.4f}")

print("\nNLL")
print(f"Vorher:  {uncalibrated_nll:.4f}")
print(f"Nachher: {calibrated_nll:.4f}")

print("\nECE")
print(f"Vorher:  {uncalibrated_ece:.4f}")
print(f"Nachher: {calibrated_ece:.4f}")

print("\nHIGH-CONFIDENCE > 0.8")
print(f"Fälle:     {cal_high_count}")
print(f"Coverage:  {cal_high_coverage:.4f}")
print(f"Accuracy:  {cal_high_accuracy:.4f}")

print("\nTOP 50 % CONFIDENCE")
print(f"Accuracy:  {cal_top50_accuracy:.4f}")

In [ ]:
# ============================================================
# LEAKAGE-CHECK: REALES FOLD-1-TRAINING vs. REALER TEST
# ============================================================

train_texts = set(
    fold1_train["clean_description"]
    .astype(str)
    .str.strip()
)

test_texts = set(
    fold1_val["clean_description"]
    .astype(str)
    .str.strip()
)

# identische Texte in Train und Test
overlap_texts = train_texts & test_texts

# Testzeilen, deren Text bereits im Training vorkommt
test_seen_mask = (
    fold1_val["clean_description"]
    .astype(str)
    .str.strip()
    .isin(overlap_texts)
)

seen_test_count = test_seen_mask.sum()
unseen_test_count = (~test_seen_mask).sum()

print("==============================================")
print("LEAKAGE-CHECK – FOLD 1")
print("==============================================")

print("Eindeutige Texte Training:", len(train_texts))
print("Eindeutige Texte Test:", len(test_texts))
print("Identische Texte in Train + Test:", len(overlap_texts))

print("\nTestfälle gesamt:", len(fold1_val))
print("Testfälle mit bereits gesehenem Text:", seen_test_count)
print("Wirklich ungesehene Testfälle:", unseen_test_count)

print(
    "Anteil bereits gesehen:",
    f"{seen_test_count / len(fold1_val):.2%}"
)

In [ ]:
# ============================================================
# PRÜFUNG: WIE VIELE EINDEUTIGE MUSTER PRO KATEGORIE?
# ============================================================

unique_patterns_per_category = (
    cv_data
    .groupby("categoryName")["clean_description"]
    .nunique()
    .sort_values()
)

print("==============================================")
print("EINDEUTIGE MUSTER PRO KATEGORIE")
print("==============================================")

display(
    unique_patterns_per_category
    .rename("unique_patterns")
    .to_frame()
)

print("\nKategorien mit weniger als 5 Mustern:")
print(
    (unique_patterns_per_category < 5).sum()
)

print("\nKategorien mit nur 1 Muster:")
print(
    (unique_patterns_per_category == 1).sum()
)

print("\nMinimum:")
print(
    unique_patterns_per_category.min()
)

In [ ]:
# ============================================================
# FINALER UNSEEN-PATTERN-HOLDOUT
# ============================================================

import numpy as np
import pandas as pd

rng = np.random.RandomState(42)

train_indices = []
test_indices = []

test_patterns = {}

for category, category_df in cv_data.groupby("categoryName"):

    unique_patterns = (
        category_df["clean_description"]
        .drop_duplicates()
        .tolist()
    )

    # Kategorie besitzt mindestens zwei unabhängige Muster
    if len(unique_patterns) >= 2:

        rng.shuffle(unique_patterns)

        # genau EIN unbekanntes Muster für den Test
        selected_test_pattern = unique_patterns[0]

        test_patterns[category] = selected_test_pattern

        category_test_idx = category_df[
            category_df["clean_description"] == selected_test_pattern
        ].index

        category_train_idx = category_df[
            category_df["clean_description"] != selected_test_pattern
        ].index

        test_indices.extend(category_test_idx)
        train_indices.extend(category_train_idx)

    else:

        # nur ein Muster vorhanden:
        # bleibt vollständig im Training
        train_indices.extend(category_df.index)


final_train_real = cv_data.loc[
    train_indices
].reset_index(drop=True)

final_test_real = cv_data.loc[
    test_indices
].reset_index(drop=True)


# ============================================================
# LEAKAGE-CHECK
# ============================================================

final_train_patterns = set(
    final_train_real["clean_description"]
)

final_test_patterns = set(
    final_test_real["clean_description"]
)

pattern_overlap = (
    final_train_patterns
    & final_test_patterns
)


print("==============================================")
print("FINALER UNSEEN-PATTERN-SPLIT")
print("==============================================")

print("Reale Trainingszeilen:", len(final_train_real))
print("Reale Testzeilen:", len(final_test_real))

print(
    "Eindeutige Trainingsmuster:",
    final_train_real["clean_description"].nunique()
)

print(
    "Eindeutige Testmuster:",
    final_test_real["clean_description"].nunique()
)

print(
    "Kategorien Training:",
    final_train_real["categoryName"].nunique()
)

print(
    "Kategorien Test:",
    final_test_real["categoryName"].nunique()
)

print(
    "Identische Muster in Train + Test:",
    len(pattern_overlap)
)

print("\nTestmuster pro Kategorie:")

display(
    pd.DataFrame(
        list(test_patterns.items()),
        columns=[
            "categoryName",
            "test_pattern"
        ]
    )
)

In [ ]:
# ============================================================
# PRÜFEN, WELCHE ALTEN SYNTHETISCHEN DATEN WIEDERVERWENDBAR SIND
# ============================================================

# Eindeutige Muster des neuen finalen Trainingssets
final_train_pattern_set = set(
    final_train_real["clean_description"]
    .astype(str)
    .str.strip()
)

final_test_pattern_set = set(
    final_test_real["clean_description"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Alte synthetische Daten:
# Nur behalten, wenn ihr SOURCE_TEXT im neuen Training liegt
# ------------------------------------------------------------

reusable_synthetic = synthetic_fold1_clean[
    synthetic_fold1_clean["source_text"]
    .astype(str)
    .str.strip()
    .isin(final_train_pattern_set)
].copy()

reusable_synthetic = reusable_synthetic.reset_index(drop=True)


# ------------------------------------------------------------
# Welche finalen Trainingsmuster haben bereits Synthetic Data?
# ------------------------------------------------------------

covered_source_patterns = set(
    reusable_synthetic["source_text"]
    .astype(str)
    .str.strip()
)

missing_source_patterns = (
    final_train_pattern_set
    - covered_source_patterns
)


# ------------------------------------------------------------
# Sicherheitscheck:
# Kein synthetisches Beispiel darf von einem Testmuster stammen
# ------------------------------------------------------------

synthetic_from_test_sources = (
    reusable_synthetic["source_text"]
    .astype(str)
    .str.strip()
    .isin(final_test_pattern_set)
    .sum()
)


print("==============================================")
print("WIEDERVERWENDUNG SYNTHETISCHER DATEN")
print("==============================================")

print(
    "Alte bereinigte synthetische Beispiele:",
    len(synthetic_fold1_clean)
)

print(
    "Davon wiederverwendbar:",
    len(reusable_synthetic)
)

print(
    "Finale Trainingsmuster insgesamt:",
    len(final_train_pattern_set)
)

print(
    "Bereits durch Synthetic Data abgedeckt:",
    len(covered_source_patterns)
)

print(
    "Noch fehlende Trainingsmuster:",
    len(missing_source_patterns)
)

print(
    "Synthetic Data aus Test-Source-Mustern:",
    synthetic_from_test_sources
)

print("\nFehlende Trainingsmuster:")

display(
    pd.DataFrame(
        sorted(missing_source_patterns),
        columns=["missing_source_text"]
    )
)

In [ ]:
# ============================================================
# NEUER BALANCING-PLAN FÜR DEN FINALEN TRAININGSSPLIT
# ============================================================

final_train_counts = (
    final_train_real["categoryName"]
    .value_counts()
    .sort_values()
)

max_count_final = final_train_counts.max()

final_augmentation_plan = pd.DataFrame({
    "real_train": final_train_counts
})

# Paper-inspirierte inverse-frequency scaling
final_augmentation_plan["synthetic_ratio"] = (
    max_count_final
    / final_augmentation_plan["real_train"]
).round()

final_augmentation_plan["synthetic_ratio"] = (
    final_augmentation_plan["synthetic_ratio"]
    .clip(lower=1, upper=30)
    .astype(int)
)

final_augmentation_plan["synthetic_needed"] = (
    final_augmentation_plan["real_train"]
    * final_augmentation_plan["synthetic_ratio"]
)

final_augmentation_plan["final_size"] = (
    final_augmentation_plan["real_train"]
    + final_augmentation_plan["synthetic_needed"]
)

print("==============================================")
print("FINALER AUGMENTATION-PLAN")
print("==============================================")

print("Reale Trainingsbeispiele:", len(final_train_real))
print(
    "Geplante synthetische Beispiele:",
    final_augmentation_plan["synthetic_needed"].sum()
)
print(
    "Gesamt nach Augmentation:",
    final_augmentation_plan["final_size"].sum()
)

display(final_augmentation_plan)

In [ ]:
# ============================================================
# EXAKTER NACHGENERIERUNGSBEDARF PRO KATEGORIE
# ============================================================

# Bereits vorhandene wiederverwendbare synthetische Beispiele
reusable_counts = (
    reusable_synthetic["categoryName"]
    .value_counts()
)

final_generation_need = final_augmentation_plan.copy()

final_generation_need["reusable_synthetic"] = (
    final_generation_need.index
    .map(reusable_counts)
    .fillna(0)
    .astype(int)
)

# Wie viele fehlen tatsächlich noch?
final_generation_need["still_needed"] = (
    final_generation_need["synthetic_needed"]
    - final_generation_need["reusable_synthetic"]
)

# Falls irgendwo mehr vorhanden als benötigt:
final_generation_need["still_needed"] = (
    final_generation_need["still_needed"]
    .clip(lower=0)
    .astype(int)
)

print("==============================================")
print("FINALER NACHGENERIERUNGSBEDARF")
print("==============================================")

print(
    "Geplante synthetische Beispiele insgesamt:",
    final_generation_need["synthetic_needed"].sum()
)

print(
    "Bereits wiederverwendbar:",
    final_generation_need["reusable_synthetic"].sum()
)

print(
    "Noch neu zu generieren:",
    final_generation_need["still_needed"].sum()
)

print(
    "Kategorien mit Nachgenerierungsbedarf:",
    (final_generation_need["still_needed"] > 0).sum()
)

display(
    final_generation_need[
        final_generation_need["still_needed"] > 0
    ][
        [
            "real_train",
            "synthetic_needed",
            "reusable_synthetic",
            "still_needed"
        ]
    ]
    .sort_values("still_needed", ascending=False)
)

In [ ]:
# ============================================================
# FINALER GENERIERUNGSPLAN FÜR DIE 2308 FEHLENDEN BEISPIELE
# ============================================================

final_generation_rows = []

rng = np.random.RandomState(42)

# Bereits abgedeckte Source-Texte
covered_sources = set(
    reusable_synthetic["source_text"]
    .astype(str)
    .str.strip()
)

for category, row in final_generation_need.iterrows():

    still_needed = int(row["still_needed"])

    # Für diese Kategorie nichts mehr nötig
    if still_needed <= 0:
        continue

    # Eindeutige reale Trainingsmuster dieser Kategorie
    category_sources = (
        final_train_real.loc[
            final_train_real["categoryName"] == category,
            "clean_description"
        ]
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    rng.shuffle(category_sources)

    # --------------------------------------------------------
    # Zuerst bisher NICHT synthetisch abgedeckte Muster
    # --------------------------------------------------------

    uncovered = [
        text
        for text in category_sources
        if text not in covered_sources
    ]

    covered = [
        text
        for text in category_sources
        if text in covered_sources
    ]

    # Uncovered zuerst
    ordered_sources = uncovered + covered

    n_sources = len(ordered_sources)

    # Gleichmäßig auf reale Muster verteilen
    base_n = still_needed // n_sources
    remainder = still_needed % n_sources

    for i, source_text in enumerate(ordered_sources):

        n_generate = base_n

        if i < remainder:
            n_generate += 1

        # Nur Zeilen anlegen, wenn wirklich etwas generiert wird
        if n_generate > 0:

            final_generation_rows.append({
                "categoryName": category,
                "source_text": source_text,
                "n_generate": n_generate,
                "previously_covered": source_text in covered_sources
            })


final_generation_plan = pd.DataFrame(
    final_generation_rows
)


# ============================================================
# KONTROLLE
# ============================================================

print("==============================================")
print("FINALER GENERIERUNGSPLAN")
print("==============================================")

print(
    "Neu zu generierende Beispiele:",
    final_generation_plan["n_generate"].sum()
)

print(
    "Verwendete reale Ausgangsmuster:",
    final_generation_plan["source_text"].nunique()
)

print(
    "Davon vorher nicht abgedeckte Ausgangsmuster:",
    (
        final_generation_plan["previously_covered"] == False
    ).sum()
)

print(
    "Kategorien:",
    final_generation_plan["categoryName"].nunique()
)

display(
    final_generation_plan.head(40)
)

In [ ]:
# ============================================================
# KORRIGIERTER FINALER SYNTHETIC-PLAN
# - exakt 7033 synthetische Beispiele
# - keine Überrepräsentation durch alte Synthetic Data
# - alle bisher fehlenden Trainingsmuster werden berücksichtigt
# ============================================================

selected_reusable_rows = []
corrected_generation_rows = []

rng = np.random.RandomState(42)

# Alle bisher durch wiederverwendbare Synthetic Data abgedeckten Sources
covered_sources = set(
    reusable_synthetic["source_text"]
    .astype(str)
    .str.strip()
)

for category, row in final_augmentation_plan.iterrows():

    target_synthetic = int(row["synthetic_needed"])

    # --------------------------------------------------------
    # Reale eindeutige Trainingsmuster dieser Kategorie
    # --------------------------------------------------------

    category_sources = (
        final_train_real.loc[
            final_train_real["categoryName"] == category,
            "clean_description"
        ]
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    rng.shuffle(category_sources)

    uncovered_sources = [
        source
        for source in category_sources
        if source not in covered_sources
    ]

    # --------------------------------------------------------
    # Wiederverwendbare synthetische Beispiele dieser Kategorie
    # --------------------------------------------------------

    category_reusable = reusable_synthetic[
        reusable_synthetic["categoryName"] == category
    ].copy()

    category_reusable = category_reusable.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Mindestens 1 neue Variante für jedes bisher
    # nicht abgedeckte reale Muster reservieren
    # --------------------------------------------------------

    minimum_new_for_coverage = len(uncovered_sources)

    # Wie viel müssten wir wegen Datenmangel ohnehin neu erzeugen?
    shortage = max(
        0,
        target_synthetic - len(category_reusable)
    )

    generation_needed = max(
        shortage,
        minimum_new_for_coverage
    )

    # Sicherheit
    generation_needed = min(
        generation_needed,
        target_synthetic
    )

    reusable_needed = (
        target_synthetic
        - generation_needed
    )

    # --------------------------------------------------------
    # Nur exakt benötigte alte Synthetic Data übernehmen
    # --------------------------------------------------------

    selected_reusable = category_reusable.head(
        reusable_needed
    ).copy()

    selected_reusable_rows.append(
        selected_reusable
    )

    # --------------------------------------------------------
    # Neue Generierung auf reale Sources verteilen
    # Uncovered Sources zuerst
    # --------------------------------------------------------

    if generation_needed > 0:

        covered_category_sources = [
            source
            for source in category_sources
            if source in covered_sources
        ]

        ordered_sources = (
            uncovered_sources
            + covered_category_sources
        )

        n_sources = len(ordered_sources)

        # Zuerst jedem bislang nicht abgedeckten Muster 1 geben
        allocation = {
            source: 0
            for source in ordered_sources
        }

        remaining = generation_needed

        for source in uncovered_sources:

            if remaining > 0:
                allocation[source] += 1
                remaining -= 1

        # Rest gleichmäßig verteilen
        i = 0

        while remaining > 0:

            source = ordered_sources[
                i % n_sources
            ]

            allocation[source] += 1
            remaining -= 1
            i += 1

        for source_text, n_generate in allocation.items():

            if n_generate > 0:

                corrected_generation_rows.append({
                    "categoryName": category,
                    "source_text": source_text,
                    "n_generate": n_generate,
                    "previously_covered":
                        source_text in covered_sources
                })


# ============================================================
# DATAFRAMES ERSTELLEN
# ============================================================

final_reusable_synthetic = pd.concat(
    selected_reusable_rows,
    ignore_index=True
)

final_generation_plan = pd.DataFrame(
    corrected_generation_rows
)


# ============================================================
# KONTROLLE
# ============================================================

reusable_total = len(
    final_reusable_synthetic
)

new_total = (
    final_generation_plan["n_generate"].sum()
)

final_total = (
    reusable_total
    + new_total
)

new_uncovered_sources = set(
    final_generation_plan.loc[
        final_generation_plan["previously_covered"] == False,
        "source_text"
    ]
)

print("==============================================")
print("KORRIGIERTER FINALER SYNTHETIC-PLAN")
print("==============================================")

print(
    "Wiederverwendbare Synthetic Data:",
    reusable_total
)

print(
    "Neu zu generieren:",
    new_total
)

print(
    "Finale Synthetic Data insgesamt:",
    final_total
)

print(
    "Neu berücksichtigte bisher fehlende Muster:",
    len(new_uncovered_sources)
)

print(
    "Kategorien im Generierungsplan:",
    final_generation_plan["categoryName"].nunique()
)

In [ ]:
# ============================================================
# FINALE NACHGENERIERUNG – 2310 SYNTHETISCHE BEISPIELE
# ============================================================

import pandas as pd
import time
import os

output_file = "final_new_synthetic.csv"

new_synthetic_rows = []

# Falls schon eine Datei existiert, laden wir sie zum Fortsetzen
if os.path.exists(output_file):

    existing_new = pd.read_csv(output_file)

    new_synthetic_rows = existing_new.to_dict(
        orient="records"
    )

    already_done_pairs = set(
        zip(
            existing_new["categoryName"],
            existing_new["source_text"]
        )
    )

    print(
        "Vorhandene neue Beispiele geladen:",
        len(existing_new)
    )

else:

    already_done_pairs = set()


# ============================================================
# GENERIERUNG
# ============================================================

for pos, (_, row) in enumerate(
    final_generation_plan.iterrows(),
    start=1
):

    category = row["categoryName"]
    source_text = row["source_text"]
    n_generate = int(row["n_generate"])

    pair = (
        category,
        source_text
    )

    # Bereits vollständig bearbeitete Source überspringen
    if pair in already_done_pairs:

        print(
            f"[{pos}/{len(final_generation_plan)}] "
            f"ÜBERSPRUNGEN | {category} | {source_text}"
        )

        continue

    print(
        f"[{pos}/{len(final_generation_plan)}] "
        f"{category} | {source_text} | "
        f"{n_generate} Varianten"
    )

    generated = generate_paper_variants(
        source_text=source_text,
        category=category,
        n=n_generate
    )

    for text in generated:

        new_synthetic_rows.append({
            "clean_description": text,
            "categoryName": category,
            "source_text": source_text,
            "synthetic": True
        })

    # --------------------------------------------------------
    # Nach jeder Source speichern
    # --------------------------------------------------------

    current_new_df = pd.DataFrame(
        new_synthetic_rows
    )

    current_new_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Neu gespeichert:",
        len(current_new_df)
    )

    time.sleep(0.2)


# ============================================================
# ABSCHLUSSKONTROLLE
# ============================================================

final_new_synthetic = pd.DataFrame(
    new_synthetic_rows
)

print("\n==============================================")
print("FINALE NACHGENERIERUNG ABGESCHLOSSEN")
print("==============================================")

print(
    "Neu erzeugte synthetische Beispiele:",
    len(final_new_synthetic)
)

print(
    "Ziel:",
    final_generation_plan["n_generate"].sum()
)

In [ ]:
# ============================================================
# FINALER QUALITÄTS- UND LEAKAGE-CHECK
# ============================================================

# Neue Daten sicher aus Datei laden
final_new_synthetic = pd.read_csv(
    "final_new_synthetic.csv"
)

# Gleiches Cleaning auf neue Daten anwenden
final_new_synthetic["clean_description"] = (
    final_new_synthetic["clean_description"]
    .astype(str)
    .apply(clean_description)
)

# Auch wiederverwendbare Daten sicher kopieren
final_reusable_synthetic_clean = (
    final_reusable_synthetic.copy()
)

final_reusable_synthetic_clean["clean_description"] = (
    final_reusable_synthetic_clean["clean_description"]
    .astype(str)
    .apply(clean_description)
)


# ============================================================
# 1. ZUSAMMENFÜHREN
# ============================================================

final_synthetic_raw = pd.concat(
    [
        final_reusable_synthetic_clean,
        final_new_synthetic
    ],
    ignore_index=True
)


# ============================================================
# 2. LEERE TEXTE
# ============================================================

empty_count = (
    final_synthetic_raw["clean_description"]
    .str.strip()
    .eq("")
    .sum()
)


# ============================================================
# 3. DUPLIKATE
# ============================================================

duplicate_count = final_synthetic_raw.duplicated(
    subset=[
        "clean_description",
        "categoryName"
    ]
).sum()


# ============================================================
# 4. OVERLAP MIT FINALEN TESTMUSTERN
# ============================================================

test_pattern_overlap = (
    final_synthetic_raw["clean_description"]
    .astype(str)
    .str.strip()
    .isin(final_test_pattern_set)
    .sum()
)


# ============================================================
# 5. SYNTHETISCHE DATEN, DEREN SOURCE AUS DEM TEST STAMMT
# ============================================================

test_source_overlap = (
    final_synthetic_raw["source_text"]
    .astype(str)
    .str.strip()
    .isin(final_test_pattern_set)
    .sum()
)


# ============================================================
# 6. KATEGORIEN
# ============================================================

category_count = (
    final_synthetic_raw["categoryName"]
    .nunique()
)


# ============================================================
# AUSGABE
# ============================================================

print("==============================================")
print("FINALER SYNTHETIC-QUALITÄTSCHECK")
print("==============================================")

print(
    "Synthetische Beispiele insgesamt:",
    len(final_synthetic_raw)
)

print(
    "Leere Beschreibungen:",
    empty_count
)

print(
    "Doppelte Beispiele:",
    duplicate_count
)

print(
    "Synthetic Text identisch mit Testmuster:",
    test_pattern_overlap
)

print(
    "Synthetic Source stammt aus Test:",
    test_source_overlap
)

print(
    "Kategorien:",
    category_count
)

In [ ]:
# ============================================================
# FINALE BEREINIGUNG + EXAKTEN ERSATZBEDARF BERECHNEN
# ============================================================

final_synthetic_clean = final_synthetic_raw.copy()


# ------------------------------------------------------------
# 1. Leere Texte entfernen
# ------------------------------------------------------------

final_synthetic_clean = final_synthetic_clean[
    final_synthetic_clean["clean_description"]
    .astype(str)
    .str.strip()
    .ne("")
].copy()


# ------------------------------------------------------------
# 2. Alle Synthetic-Texte entfernen,
#    die exakt einem finalen Testmuster entsprechen
# ------------------------------------------------------------

final_synthetic_clean = final_synthetic_clean[
    ~final_synthetic_clean["clean_description"]
    .astype(str)
    .str.strip()
    .isin(final_test_pattern_set)
].copy()


# ------------------------------------------------------------
# 3. Duplikate innerhalb derselben Kategorie entfernen
# ------------------------------------------------------------

final_synthetic_clean = (
    final_synthetic_clean
    .drop_duplicates(
        subset=[
            "clean_description",
            "categoryName"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 4. IST-BESTAND PRO KATEGORIE
# ============================================================

clean_counts = (
    final_synthetic_clean["categoryName"]
    .value_counts()
)


# ============================================================
# 5. MIT SOLL-MENGE AUS AUGMENTATION-PLAN VERGLEICHEN
# ============================================================

replacement_plan = final_augmentation_plan[
    ["synthetic_needed"]
].copy()

replacement_plan["current_clean"] = (
    replacement_plan.index
    .map(clean_counts)
    .fillna(0)
    .astype(int)
)

replacement_plan["replacement_needed"] = (
    replacement_plan["synthetic_needed"]
    - replacement_plan["current_clean"]
)

replacement_plan["replacement_needed"] = (
    replacement_plan["replacement_needed"]
    .clip(lower=0)
    .astype(int)
)


# ============================================================
# 6. SICHERHEITSCHECKS
# ============================================================

remaining_test_overlap = (
    final_synthetic_clean["clean_description"]
    .astype(str)
    .str.strip()
    .isin(final_test_pattern_set)
    .sum()
)

remaining_duplicates = (
    final_synthetic_clean
    .duplicated(
        subset=[
            "clean_description",
            "categoryName"
        ]
    )
    .sum()
)


print("==============================================")
print("BEREINIGTER FINALER SYNTHETIC-DATENSATZ")
print("==============================================")

print(
    "Vor Bereinigung:",
    len(final_synthetic_raw)
)

print(
    "Nach Bereinigung:",
    len(final_synthetic_clean)
)

print(
    "Insgesamt entfernt:",
    len(final_synthetic_raw)
    - len(final_synthetic_clean)
)

print(
    "Verbleibende Duplikate:",
    remaining_duplicates
)

print(
    "Verbleibender Test-Overlap:",
    remaining_test_overlap
)

print(
    "Kategorien:",
    final_synthetic_clean["categoryName"].nunique()
)

print("\n==============================================")
print("ERSATZBEDARF")
print("==============================================")

print(
    "Noch neu zu generieren:",
    replacement_plan["replacement_needed"].sum()
)

print(
    "Kategorien mit Ersatzbedarf:",
    (replacement_plan["replacement_needed"] > 0).sum()
)

display(
    replacement_plan[
        replacement_plan["replacement_needed"] > 0
    ]
    .sort_values(
        "replacement_needed",
        ascending=False
    )
)

In [ ]:
# ============================================================
# ERSATZGENERATION FÜR DIE 477 FEHLENDEN BEISPIELE
# MIT DIREKTER DUPLIKAT- UND LEAKAGE-PRÜFUNG
# ============================================================

import pandas as pd

replacement_rows = []

# Bereits vorhandene finale synthetische Texte pro Kategorie
existing_pairs = set(
    zip(
        final_synthetic_clean["clean_description"],
        final_synthetic_clean["categoryName"]
    )
)

# Testmuster als Set
test_patterns_set = set(
    final_test_pattern_set
)


for category, row in replacement_plan.iterrows():

    needed = int(row["replacement_needed"])

    if needed <= 0:
        continue

    # Reale Trainingsmuster dieser Kategorie
    source_candidates = (
        final_train_real.loc[
            final_train_real["categoryName"] == category,
            "clean_description"
        ]
        .astype(str)
        .str.strip()
        .drop_duplicates()
        .tolist()
    )

    print(
        f"\nKategorie: {category} | "
        f"noch benötigt: {needed}"
    )

    collected = 0
    source_idx = 0

    while collected < needed:

        source_text = source_candidates[
            source_idx % len(source_candidates)
        ]

        remaining = needed - collected

        # Etwas mehr anfordern, weil manche Varianten
        # später verworfen werden können
        request_n = min(
            max(remaining + 5, 10),
            100
        )

        generated = generate_paper_variants(
            source_text=source_text,
            category=category,
            n=request_n
        )

        for text in generated:

            cleaned = clean_description(
                str(text)
            ).strip()

            pair = (
                cleaned,
                category
            )

            # Leere Texte verwerfen
            if cleaned == "":
                continue

            # Exakte Testmuster verwerfen
            if cleaned in test_patterns_set:
                continue

            # Bereits vorhandene Kombination verwerfen
            if pair in existing_pairs:
                continue

            replacement_rows.append({
                "clean_description": cleaned,
                "categoryName": category,
                "source_text": source_text,
                "synthetic": True
            })

            existing_pairs.add(pair)

            collected += 1

            if collected >= needed:
                break

        source_idx += 1

        print(
            f"  gesammelt: {collected}/{needed}"
        )


# ============================================================
# ERSATZ-DATAFRAME
# ============================================================

replacement_synthetic = pd.DataFrame(
    replacement_rows
)

print("\n==============================================")
print("ERSATZGENERATION ABGESCHLOSSEN")
print("==============================================")

print(
    "Neu erzeugte Ersatzbeispiele:",
    len(replacement_synthetic)
)

print(
    "Ziel:",
    replacement_plan["replacement_needed"].sum()
)

In [ ]:
# ============================================================
# FINALEN SYNTHETISCHEN DATENSATZ ERSTELLEN UND PRÜFEN
# ============================================================

final_synthetic = pd.concat(
    [
        final_synthetic_clean,
        replacement_synthetic
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Sicherheitschecks
# ------------------------------------------------------------

final_duplicates = final_synthetic.duplicated(
    subset=[
        "clean_description",
        "categoryName"
    ]
).sum()

final_test_overlap = (
    final_synthetic["clean_description"]
    .astype(str)
    .str.strip()
    .isin(final_test_pattern_set)
    .sum()
)

final_test_source_overlap = (
    final_synthetic["source_text"]
    .astype(str)
    .str.strip()
    .isin(final_test_pattern_set)
    .sum()
)

final_empty = (
    final_synthetic["clean_description"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("==============================================")
print("FINALER SYNTHETIC-DATENSATZ")
print("==============================================")

print(
    "Synthetische Beispiele:",
    len(final_synthetic)
)

print(
    "Kategorien:",
    final_synthetic["categoryName"].nunique()
)

print(
    "Leere Beschreibungen:",
    final_empty
)

print(
    "Duplikate:",
    final_duplicates
)

print(
    "Synthetic Text identisch mit Testmuster:",
    final_test_overlap
)

print(
    "Synthetic Source stammt aus Test:",
    final_test_source_overlap
)


# ------------------------------------------------------------
# Pro Kategorie Soll vs. Ist prüfen
# ------------------------------------------------------------

final_category_counts = (
    final_synthetic["categoryName"]
    .value_counts()
)

final_distribution_check = final_augmentation_plan[
    ["synthetic_needed"]
].copy()

final_distribution_check["actual_synthetic"] = (
    final_distribution_check.index
    .map(final_category_counts)
    .fillna(0)
    .astype(int)
)

final_distribution_check["difference"] = (
    final_distribution_check["actual_synthetic"]
    - final_distribution_check["synthetic_needed"]
)

print("\nAbweichung von der geplanten Verteilung:")

print(
    "Summe absolute Abweichung:",
    final_distribution_check["difference"]
    .abs()
    .sum()
)

display(
    final_distribution_check[
        final_distribution_check["difference"] != 0
    ]
)


# ------------------------------------------------------------
# Speichern
# ------------------------------------------------------------

final_synthetic.to_csv(
    "final_synthetic_clean_7033.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "\nGespeichert als: final_synthetic_clean_7033.csv"
)

In [ ]:
# ============================================================
# FINALER SYNTHETIC TRAIN/VAL-SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

final_synthetic_model_data = final_synthetic[
    ["clean_description", "categoryName"]
].copy()

final_synthetic_model_data["label"] = cv_label_encoder.transform(
    final_synthetic_model_data["categoryName"]
)

final_syn_train, final_syn_val = train_test_split(
    final_synthetic_model_data,
    test_size=0.20,
    random_state=42,
    stratify=final_synthetic_model_data["label"]
)

final_syn_train = final_syn_train.reset_index(drop=True)
final_syn_val = final_syn_val.reset_index(drop=True)

print("==============================================")
print("FINALER SYNTHETIC TRAIN/VAL-SPLIT")
print("==============================================")

print("Training:", len(final_syn_train))
print("Validation:", len(final_syn_val))

print(
    "Kategorien Training:",
    final_syn_train["categoryName"].nunique()
)

print(
    "Kategorien Validation:",
    final_syn_val["categoryName"].nunique()
)

print(
    "Finale reale Testfälle:",
    len(final_test_real)
)

print(
    "Finale reale Testmuster:",
    final_test_real["clean_description"].nunique()
)

In [ ]:
# ============================================================
# FINALES FINBERT-SETUP – LEAKAGE-FREIER UNSEEN-PATTERN-LAUF
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ------------------------------------------------------------
# 1. FinBERT frisch laden
# ------------------------------------------------------------

final_model_name = "ProsusAI/finbert"

final_tokenizer = AutoTokenizer.from_pretrained(
    final_model_name
)

final_finbert = AutoModelForSequenceClassification.from_pretrained(
    final_model_name,
    num_labels=37,
    ignore_mismatched_sizes=True
)

final_finbert.to(device)

print("Frisches FinBERT geladen.")


# ------------------------------------------------------------
# 2. Synthetisches Training tokenisieren
# ------------------------------------------------------------

final_train_encodings = final_tokenizer(
    final_syn_train["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)


# ------------------------------------------------------------
# 3. Synthetische Validation tokenisieren
# ------------------------------------------------------------

final_val_encodings = final_tokenizer(
    final_syn_val["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)


# ------------------------------------------------------------
# 4. Reale UNSEEN-Testdaten tokenisieren
# ------------------------------------------------------------

final_test_encodings = final_tokenizer(
    final_test_real["clean_description"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)


# ------------------------------------------------------------
# 5. PyTorch-Datasets
# ------------------------------------------------------------

final_train_dataset = TransactionDataset(
    final_train_encodings,
    final_syn_train["label"].tolist()
)

final_val_dataset = TransactionDataset(
    final_val_encodings,
    final_syn_val["label"].tolist()
)

final_test_dataset = TransactionDataset(
    final_test_encodings,
    final_test_real["label"].tolist()
)


# ------------------------------------------------------------
# Kontrolle
# ------------------------------------------------------------

print("\n==============================================")
print("FINALES FINBERT-SETUP")
print("==============================================")

print(
    "Synthetisches Training:",
    len(final_train_dataset)
)

print(
    "Synthetische Validation:",
    len(final_val_dataset)
)

print(
    "Realer Unseen-Test:",
    len(final_test_dataset)
)

print(
    "Unabhängige reale Testmuster:",
    final_test_real["clean_description"].nunique()
)

In [ ]:
# ============================================================
# FINALE TRAININGSKONFIGURATION – UNSEEN-PATTERN-LAUF
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup


# ------------------------------------------------------------
# 1. Klassen-Gewichte aus dem FINALEN synthetischen Training
# ------------------------------------------------------------

num_labels = 37

final_train_labels_np = final_syn_train["label"].to_numpy()

final_class_counts = np.bincount(
    final_train_labels_np,
    minlength=num_labels
)

final_class_weights = (
    len(final_train_labels_np)
    /
    (num_labels * final_class_counts)
)

final_class_weights_tensor = torch.tensor(
    final_class_weights,
    dtype=torch.float32,
    device=device
)

print("Klassen-Gewichte erstellt.")
print(
    "Min. Gewicht:",
    final_class_weights_tensor.min().item()
)
print(
    "Max. Gewicht:",
    final_class_weights_tensor.max().item()
)


# ------------------------------------------------------------
# 2. Weighted Focal Loss
# ------------------------------------------------------------

class FinalWeightedFocalLoss(nn.Module):

    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        probs = F.softmax(
            logits,
            dim=1
        )

        pt = probs.gather(
            1,
            targets.unsqueeze(1)
        ).squeeze(1)

        pt = pt.clamp(
            min=1e-8,
            max=1.0
        )

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
        else:
            alpha_t = 1.0

        loss = (
            -alpha_t
            * (1 - pt) ** self.gamma
            * torch.log(pt)
        )

        return loss.mean()


final_focal_loss = FinalWeightedFocalLoss(
    alpha=final_class_weights_tensor,
    gamma=2.0
)


# ------------------------------------------------------------
# 3. DataLoader
# ------------------------------------------------------------

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=16,
    shuffle=True
)

final_val_loader = DataLoader(
    final_val_dataset,
    batch_size=16,
    shuffle=False
)

final_test_loader = DataLoader(
    final_test_dataset,
    batch_size=16,
    shuffle=False
)


# ------------------------------------------------------------
# 4. Optimizer
# ------------------------------------------------------------

final_optimizer = torch.optim.AdamW(
    final_finbert.parameters(),
    lr=2e-5
)


# ------------------------------------------------------------
# 5. Scheduler
# ------------------------------------------------------------

final_epochs = 3

final_total_steps = (
    len(final_train_loader)
    * final_epochs
)

final_warmup_steps = 500

final_scheduler = get_linear_schedule_with_warmup(
    final_optimizer,
    num_warmup_steps=final_warmup_steps,
    num_training_steps=final_total_steps
)


# ------------------------------------------------------------
# Kontrolle
# ------------------------------------------------------------

print("\n==============================================")
print("FINALE TRAININGSKONFIGURATION")
print("==============================================")

print("Trainingsbeispiele:", len(final_train_dataset))
print("Validation:", len(final_val_dataset))
print("Realer Unseen-Test:", len(final_test_dataset))
print("Batch Size: 16")
print("Learning Rate: 2e-5")
print("Epochs:", final_epochs)
print("Batches pro Epoche:", len(final_train_loader))
print("Gesamte Trainingsschritte:", final_total_steps)
print("Warmup Steps:", final_warmup_steps)
print("Focal Loss Gamma: 2.0")

In [ ]:
# ============================================================
# FINALES FINBERT-TRAINING – UNSEEN-PATTERN-LAUF
# ============================================================

from tqdm.auto import tqdm
import copy

final_best_val_accuracy = 0.0
final_best_model_state = None
final_training_history = []

for epoch in range(final_epochs):

    # ========================================================
    # TRAINING
    # ========================================================

    final_finbert.train()

    total_train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_progress = tqdm(
        final_train_loader,
        desc=f"Epoch {epoch + 1}/{final_epochs} - Training"
    )

    for batch in train_progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        final_optimizer.zero_grad()

        outputs = final_finbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        loss = final_focal_loss(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            final_finbert.parameters(),
            max_norm=1.0
        )

        final_optimizer.step()
        final_scheduler.step()

        total_train_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

        train_progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = (
        total_train_loss
        / len(final_train_loader)
    )

    train_accuracy = (
        train_correct
        / train_total
    )

    # ========================================================
    # SYNTHETISCHE VALIDATION
    # ========================================================

    final_finbert.eval()

    total_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch in final_val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = final_finbert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            loss = final_focal_loss(
                logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                logits,
                dim=1
            )

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_loss = (
        total_val_loss
        / len(final_val_loader)
    )

    val_accuracy = (
        val_correct
        / val_total
    )

    # ========================================================
    # BESTES MODELL SPEICHERN
    # ========================================================

    if val_accuracy > final_best_val_accuracy:

        final_best_val_accuracy = val_accuracy

        final_best_model_state = copy.deepcopy(
            final_finbert.state_dict()
        )

    final_training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })

    print("\n==============================================")
    print(f"EPOCH {epoch + 1}/{final_epochs}")
    print("==============================================")
    print(f"Train Loss:     {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss:       {val_loss:.4f}")
    print(f"Val Accuracy:   {val_accuracy:.4f}")
    print("==============================================\n")


# ============================================================
# BESTES MODELL WIEDERHERSTELLEN
# ============================================================

final_finbert.load_state_dict(
    final_best_model_state
)

print("FINALES TRAINING ABGESCHLOSSEN")
print(
    f"Beste synthetische Validation Accuracy: "
    f"{final_best_val_accuracy:.4f}"
)

In [ ]:
# ============================================================
# FINALE REALE UNSEEN-PATTERN-EVALUATION – UNKALIBRIERT
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

final_finbert.eval()

final_test_logits_list = []
final_test_labels_list = []

with torch.no_grad():

    for batch in final_test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = final_finbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        final_test_logits_list.append(
            outputs.logits.cpu()
        )

        final_test_labels_list.append(
            labels.cpu()
        )


# ------------------------------------------------------------
# Zusammenführen
# ------------------------------------------------------------

final_test_logits = torch.cat(
    final_test_logits_list,
    dim=0
)

final_test_labels = torch.cat(
    final_test_labels_list,
    dim=0
).numpy()


# ------------------------------------------------------------
# Wahrscheinlichkeiten
# ------------------------------------------------------------

final_test_probs = torch.softmax(
    final_test_logits,
    dim=1
).numpy()

final_test_predictions = np.argmax(
    final_test_probs,
    axis=1
)

final_test_confidence = np.max(
    final_test_probs,
    axis=1
)


# ------------------------------------------------------------
# Standardmetriken
# ------------------------------------------------------------

final_test_accuracy = accuracy_score(
    final_test_labels,
    final_test_predictions
)

final_test_macro_f1 = f1_score(
    final_test_labels,
    final_test_predictions,
    average="macro",
    zero_division=0
)

final_test_weighted_f1 = f1_score(
    final_test_labels,
    final_test_predictions,
    average="weighted",
    zero_division=0
)


# ------------------------------------------------------------
# Top-2 Accuracy
# ------------------------------------------------------------

final_test_top2 = np.argsort(
    final_test_probs,
    axis=1
)[:, -2:]

final_test_top2_accuracy = np.mean(
    [
        true_label in top2
        for true_label, top2
        in zip(
            final_test_labels,
            final_test_top2
        )
    ]
)


# ------------------------------------------------------------
# High Confidence > 0.8
# ------------------------------------------------------------

final_high_mask = (
    final_test_confidence > 0.8
)

final_high_count = (
    final_high_mask.sum()
)

final_high_coverage = (
    final_high_mask.mean()
)

if final_high_count > 0:

    final_high_accuracy = accuracy_score(
        final_test_labels[final_high_mask],
        final_test_predictions[final_high_mask]
    )

else:

    final_high_accuracy = np.nan


# ------------------------------------------------------------
# Top 50 % Confidence
# ------------------------------------------------------------

final_top50_count = int(
    len(final_test_confidence) * 0.50
)

final_top50_indices = np.argsort(
    final_test_confidence
)[-final_top50_count:]

final_top50_accuracy = accuracy_score(
    final_test_labels[final_top50_indices],
    final_test_predictions[final_top50_indices]
)


# ------------------------------------------------------------
# Ausgabe
# ------------------------------------------------------------

print("==============================================")
print("FINALE REALE UNSEEN-EVALUATION – UNKALIBRIERT")
print("==============================================")

print(f"Reale Testfälle:               {len(final_test_labels)}")
print(f"Unabhängige Testmuster:        {final_test_real['clean_description'].nunique()}")
print(f"Standard Accuracy:             {final_test_accuracy:.4f}")
print(f"Macro F1:                      {final_test_macro_f1:.4f}")
print(f"Weighted F1:                   {final_test_weighted_f1:.4f}")
print(f"Top-2 Accuracy:                {final_test_top2_accuracy:.4f}")

print("\n--- Confidence ---")

print(f"> 0.8 Fälle:                    {final_high_count}")
print(f"> 0.8 Coverage:                 {final_high_coverage:.4f}")
print(f"> 0.8 Accuracy:                 {final_high_accuracy:.4f}")
print(f"Top-50%-Confidence Accuracy:   {final_top50_accuracy:.4f}")

In [ ]:
# ============================================================
# ANALYSE DER 33 UNABHÄNGIGEN TESTMUSTER
# ============================================================

pattern_results = final_test_real[
    ["clean_description", "categoryName", "label"]
].copy()

pattern_results["predicted_label"] = final_test_predictions
pattern_results["confidence"] = final_test_confidence

pattern_results["predicted_category"] = (
    cv_label_encoder.inverse_transform(
        pattern_results["predicted_label"].to_numpy()
    )
)

pattern_results["correct"] = (
    pattern_results["label"]
    == pattern_results["predicted_label"]
)

# Da viele Testzeilen dasselbe Muster wiederholen:
# pro unabhängigem Muster genau eine Zeile betrachten
pattern_level_results = (
    pattern_results
    .drop_duplicates(
        subset=["clean_description", "categoryName"]
    )
    .reset_index(drop=True)
)

pattern_accuracy = (
    pattern_level_results["correct"].mean()
)

print("==============================================")
print("ERGEBNIS AUF UNABHÄNGIGEN TESTMUSTERN")
print("==============================================")

print(
    "Unabhängige Muster:",
    len(pattern_level_results)
)

print(
    "Richtig klassifizierte Muster:",
    pattern_level_results["correct"].sum()
)

print(
    "Falsch klassifizierte Muster:",
    (~pattern_level_results["correct"]).sum()
)

print(
    f"Muster-Level Accuracy: {pattern_accuracy:.4f}"
)

print("\n==============================================")
print("FALSCH KLASSIFIZIERTE MUSTER")
print("==============================================")

display(
    pattern_level_results.loc[
        ~pattern_level_results["correct"],
        [
            "clean_description",
            "categoryName",
            "predicted_category",
            "confidence"
        ]
    ].sort_values(
        "confidence",
        ascending=False
    )
)

In [ ]:
# ============================================================
# CONFIDENCE-THRESHOLDS AUF UNABHÄNGIGEN TESTMUSTERN
# ============================================================

thresholds = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]

threshold_results = []

for threshold in thresholds:

    mask = (
        pattern_level_results["confidence"]
        >= threshold
    )

    count = mask.sum()

    coverage = (
        count
        / len(pattern_level_results)
    )

    if count > 0:

        accuracy = (
            pattern_level_results.loc[
                mask,
                "correct"
            ].mean()
        )

    else:

        accuracy = np.nan

    threshold_results.append({
        "threshold": threshold,
        "cases": count,
        "coverage": coverage,
        "accuracy": accuracy
    })


threshold_results_df = pd.DataFrame(
    threshold_results
)

print("==============================================")
print("CONFIDENCE-THRESHOLDS – UNSEEN PATTERNS")
print("==============================================")

display(
    threshold_results_df
)